In [ ]:
import faulthandler, glob, json, os, platform, subprocess, sys, threading, time
T0 = time.time()
faulthandler.enable(all_threads=True)   # a segfault / abort prints every thread's traceback (commit v1 died without one: an OOM kill prints nothing)


def monitor(every=120):                 # process RSS, MemAvailable, GPU memory (nvidia-smi: no CUDA call from this thread)
    while True:
        time.sleep(every)
        try:
            rss = int(next(l for l in open("/proc/self/status") if l.startswith("VmRSS")).split()[1]) / 2**20
            av = int(next(l for l in open("/proc/meminfo") if l.startswith("MemAvailable")).split()[1]) / 2**20
            gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True,
                                 timeout=20).stdout.split()
            print(f"[mon {time.time() - T0:.0f}s] rss {rss:.1f} GB, MemAvailable {av:.1f} GB, gpu used {gpu} MiB", flush=True)
        except Exception as e:
            print(f"[mon] {type(e).__name__}: {e}", flush=True)


threading.Thread(target=monitor, daemon=True).start()
import numpy as np, pandas as pd, torch
SLUG = 'knee-oct13-f5-win'
INPUT = os.environ.get("KNEE_INPUT", "/kaggle/input")  # local dry run: a folder laid out like /kaggle/input
COMP = next(p for p in [INPUT + "/competitions/rsna-knee-abnormality-detection", INPUT + "/rsna-knee-abnormality-detection"]
            if os.path.exists(p + "/test.csv"))
PKG = next(os.path.dirname(p) for pat in ["/*/config.json", "/*/*/config.json", "/*/*/*/config.json"]
           for p in sorted(glob.glob(INPUT + pat)) if SLUG in p)
def _find(slug, fname):
    return next(os.path.dirname(p) for pat in ["/*/" + fname, "/*/*/" + fname, "/*/*/*/" + fname] for p in sorted(glob.glob(INPUT + pat)) if slug in p)


ZMEM = _find('knee-oct17-mcl-zoom', "member.json")                  # Q21: the MCL zoom member w5b_mcl (+ mcl_member)
SDM = _find('knee-oct18-mcl-sd2', "member_sd.json")               # Q21: SD-2 (5 main SD + 5 NFNet SD fold weights + the zoom re-fit)
AMEM = _find('knee-oct18-acl-b3g', "member.json")                 # Q16: z1_acl + b3g (b3g's files = the MCL-stack dataset's, md5-equal)
LMEM = _find('knee-oct18-loa-allup', "member.json")                 # Q15: oasep / oasep_nf / silsep OA groups + loa_set / align
CMEM = _find('knee-oct16-syn-cnx-probe', "syn_members.json")            # Syn crops: the 6 ConvNeXt crop members (cs_*) of 57283240's package
CFM = _find('knee-oct17-cont-fs', "member.json")                  # Contusion FS: bfs_fs (5 fold bone models, nofuse) + cont_fs constants
MFM = _find('knee-oct19-frac-mfs', "member.json")                  # SWITCH frac_mfs: m_fs (5 fsmain fold models) + frac_mfs constants
BKM = _find('knee-oct19-bakers-allpw', "member.json")                  # SWITCH bakers_allpw: pw main f0-4 + pw NFNet / R50 basin f0-4 (T_eval 16 json)
SWITCHES = {'frac_mfs': False, 'pf_q20': False, 'bakers_allpw': False, 'lm_allpw': False}
PF_Q20 = {'set': ['oa_oasep', 'oa_oasep_nf'], 'align': {'mu_f': -1.0267274179181183, 'sd_f': 1.5497894554004341, 'mu_e': -1.1916145362529003, 'sd_e': 1.386014327393857}, 'col': 'PF OA'}                                     # SWITCH pf_q20: oct18_probe_pf_allup's loa_set / align (its members = Q15's, md5-equal)
STACK = {'name': 'b3g', 'w': 0.25, 'mu_m': -2.5089040642103666, 'sd_m': 0.7867673327816351, 'mu_d': -2.5535264494255654, 'sd_d': 0.8051382978062311, 'n_align': 2584}                                       # Q21: mcl_member2 = b3g aligned to the SD-2 + zoom MCL
# oct19_stack_fold (Experts V3 10-18, Master; USER: measured + pruned, NEVER submitted): the 0.961 fold package for 7 columns (+ LM in variant a)
# and the four LB-won column forms - MCL = Q21, LOA = Q15, ACL = Q16, Synovitis = the ConvNeXt crop slots - each by its probe's own code
# path (config keys on disjoint columns). FULL form: b3g runs twice as in the probes (b3g_mcl keep mcl, b3g_acl keep notch; same weights);
# prune variant P1 (B3G_SHARED): ONE b3g pass (keep_bags 'mcl,notch') serves both.
B3G_SHARED = True
P2_EFF = None   # prune variant P2 (C125): Effusion's img_eff 'main' pipeline from the ConvNeXt crop members (img_eff stats re-fitted label-free)
WORK = "/kaggle/working/pkg_stack" if os.path.isdir("/kaggle/working") else os.path.join(os.environ.get("TMPDIR", "/tmp"), "pkg_stack")
os.makedirs(WORK, exist_ok=True)
import shutil
PATCHED = {"syn_stage.py": '"""oct18_probe_acl_b3g (Experts V3, 2026-10-18; = oct17_probe_acl_zoom\'s file + member cfg \'keep_bags\' in study_tensors / CROP_KEYS): the 0.961 package\'s syn_stage.py + additive changes for one extra crop member that is\nnot a synovitis expert (the MCL zoom w5b_mcl / the ACL notch40 zoom z1_acl of experiments/v3/oct16_experts): bag tables = that experiment\'s syn_data.py (adds mclz and\nthe other region / zoom views; the deployed bags keep their entries), per-output view routing (config out_bag), img_syn optional, every\ntext output returned as \'<expert>@<out>\'. The deployed members\' path is unchanged (same code for them).\nTest-time path of the synovitis crop experts (main/syn_data.py + main/syn_train.py SynNet), adapted from the OA stage\n(submissions/oct11_oa_e3/oa_stage.py: localiser, series boxes, mirror detector). Flat: no experiment imports, so it can be copied\ninto a Kaggle package as is.\n\nPer study:\n  1. knee_infer.study_series: the CPU path, byte-identical to the training cache fov160_px400 (uint8 (n, 400, 400), 0.4 mm/px);\n  2. Basin\'s mirror detector (p > 0.9) per series, flipped back like the training flip table (sagittal: stack reversed, pos -> -pos;\n     coronal / axial: columns mirrored): the experts were trained in the side-corrected view;\n  3. localiser v2 (16 structures) on every slice -> series boxes (presence-weighted mean over slices with p > 0.5) -> bags with the\n     training rules of each expert\'s config (bags, planes, fs_only, trim, crop_scale, plane_bags, clamp);\n  4. each expert = mean img_Synovitis logit over its fold models (k_eval windows per bag, evenly spread).\npredict() -> {expert name: Series of synovitis logits} + {expert name@img_eff: Series of image-read Effusion logits} (oct13_f5_eff, experts\nwith an img_eff output); studies that fail are missing (logged)."""\nimport collections\nimport math\nimport os\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport timm\nimport torch\nimport torch.nn as nn\n\nimport knee_infer as ki\nfrom basin_flipdet import FlipNet, det_input\n\nSTRUCTS_V2 = ["comp_medial", "comp_lateral", "pf_joint", "patella", "joint", "med_tib_cart", "lat_tib_cart", "pat_cart", "fem_cart", "med_men",\n              "lat_men", "notch", "mcl", "suprapatellar", "hoffa", "popliteal"]\nSTRUCT = {"pouch": "suprapatellar", "hoffa": "hoffa", "post": "popliteal", "notch": "notch", "knee": "joint", "gutter": "patella",\n          "mcl": "mcl", "notchsc": "notch",           # oct16_experts: ligament expert bags (mcl: coronal + axial MCL box; notchsc: notch sag + cor)\n          "patcart": "pat_cart", "pfj": "pf_joint", "latmen": "lat_men", "medmen": "med_men", "postz": "popliteal",\n          "lattib": "lat_tib_cart", "mclz": "mcl",    # oct16_experts W batch (user U8): region zooms for PF / menisci / Baker\'s / LOA / MCL\n          "medc": "comp_medial", "latc": "comp_lateral", "pfl": "pf_joint",   # ME1 merged multi-view (user U11): 100 mm compartment / PF locals\n          "medc60": "comp_medial", "latc60": "comp_lateral", "pfl60": "pf_joint", "medc40": "comp_medial", "latc40": "comp_lateral",\n          "pfl40": "pf_joint", "notch40": "notch", "knee140": "joint"}\nPLANES_OF = {"pouch": ("Sagittal", "Coronal", "Axial"), "hoffa": ("Sagittal", "Coronal", "Axial"), "post": ("Sagittal", "Axial"),\n             "notch": ("Sagittal", "Coronal", "Axial"), "knee": ("Sagittal", "Coronal", "Axial"), "gutter": ("Axial",),\n             "mcl": ("Coronal", "Axial"), "notchsc": ("Sagittal", "Coronal"),\n             "patcart": ("Axial", "Sagittal"), "pfj": ("Axial", "Sagittal"), "latmen": ("Sagittal", "Coronal"), "medmen": ("Sagittal", "Coronal"),\n             "postz": ("Axial", "Sagittal"), "lattib": ("Coronal", "Sagittal"), "mclz": ("Coronal",),\n             "medc": ("Coronal", "Sagittal"), "latc": ("Coronal", "Sagittal"), "pfl": ("Sagittal", "Axial"),\n             "medc60": ("Coronal", "Sagittal"), "latc60": ("Coronal", "Sagittal"), "pfl60": ("Sagittal", "Axial"),\n             "medc40": ("Coronal", "Sagittal"), "latc40": ("Coronal", "Sagittal"), "pfl40": ("Sagittal", "Axial"), "notch40": ("Sagittal", "Coronal"),\n             "knee140": ("Sagittal", "Coronal", "Axial")}\nCROP_MM = {"pouch": 80, "hoffa": 72, "post": 100, "notch": 64, "knee": 160, "gutter": 110, "mcl": 70, "notchsc": 64,\n           "patcart": 48, "pfj": 72, "latmen": 48, "medmen": 48, "postz": 64, "lattib": 48, "mclz": 48, "medc": 100, "latc": 100, "pfl": 100,\n           "medc60": 60, "latc60": 60, "pfl60": 60, "medc40": 40, "latc40": 40, "pfl40": 40, "notch40": 40, "knee140": 140}\nPLANE_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}\nP_MIN, TOP_FALLBACK, LOC_PX, MM_PER_PX, PX = 0.5, 3, 128, 0.4, 400\nAMP = getattr(torch, os.environ.get("SYN_AMP_DTYPE", "float16"))   # T4: fp16; local checks may set bfloat16 = the experiment\nSTATS = {}\n\n\ndef sincos(pos, n_freq=8, scale=60.0):\n    f = torch.exp2(torch.arange(n_freq, device=pos.device, dtype=pos.dtype)) * (2 * math.pi / scale)\n    a = pos[..., None] * f\n    return torch.cat([a.sin(), a.cos()], -1)\n\n\nclass Loc(nn.Module):\n    """experiments/v1/oct10_oa/scripts/loc_train.py Loc (MobileNetV3, 3-slice 128 px windows -> presence logit + box per structure)."""\n    def __init__(self, arch="mobilenetv3_large_100", structs=STRUCTS_V2):\n        super().__init__()\n        self.n_s = len(structs)\n        self.backbone = timm.create_model(arch, pretrained=False, num_classes=0)\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.head = nn.Sequential(nn.Linear(d + 5, 256), nn.ReLU(), nn.Linear(256, self.n_s * 5))\n        c = self.backbone.pretrained_cfg\n        self.register_buffer("mean", torch.tensor(c["mean"]).view(1, 3, 1, 1) * 255)\n        self.register_buffer("std", torch.tensor(c["std"]).view(1, 3, 1, 1) * 255)\n\n    def forward(self, x, aux):\n        x = ((x.float() - self.mean) / self.std).contiguous(memory_format=torch.channels_last)\n        o = self.head(torch.cat([self.backbone(x), aux], 1)).view(-1, self.n_s, 5)\n        return o[..., 0], torch.sigmoid(o[..., 1:])\n\n\ndef aux_features(plane, pos, n):\n    p = np.zeros((n, 3), np.float32)\n    p[:, PLANE_ID[plane]] = 1\n    return np.concatenate([p, (np.asarray(pos, np.float32) / 60)[:, None], (np.arange(n) / max(n - 1, 1)).astype(np.float32)[:, None]], 1)\n\n\nclass SynNet(nn.Module):\n    """main/syn_train.py SynNet (per-output attention pooling over the allowed bags; bagsum: per-bag readout)."""\n    def __init__(self, backbone, out_bags, n_bags, n_types, hidden=256, bagsum=False):\n        super().__init__()\n        self.bagsum, self.n_bags = bagsum, n_bags\n        if backbone == "radimagenet_resnet50":         # training: timm resnet50 + RadImageNet weights, ImageNet mean / std, mono input\n            self.backbone = timm.create_model("resnet50", pretrained=False, num_classes=0)\n            self.backbone.pretrained_cfg = dict(self.backbone.pretrained_cfg, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))\n        else:\n            self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.type_emb, self.plane_emb, self.bag_emb, self.pos_proj = nn.Embedding(n_types, d), nn.Embedding(3, d), nn.Embedding(n_bags, d), nn.Linear(16, d)\n        self.norm = nn.LayerNorm(d)\n        n_out = out_bags.shape[0]\n        self.att_v, self.att_w = nn.Linear(d, hidden), nn.Linear(hidden, n_out)\n        self.cls_w = nn.Parameter(torch.zeros(n_out, n_bags, d) if bagsum else torch.zeros(n_out, d))\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        self.register_buffer("out_bags", torch.as_tensor(out_bags).T.contiguous(), persistent=False)\n\n    def forward(self, x, mask, typ, plane, pos, bag, crops=False):\n        """crops=True (oct13_f5_win, no bagsum): per-crop attention logits s and logits u (b, k, n_out); study logit = sum_k\n        softmax_k(s) u."""\n        b, k = mask.shape\n        f = self.backbone(x).float().view(b, k, -1)\n        h = self.norm(f + self.type_emb(typ) + self.plane_emb(plane) + self.bag_emb(bag.clamp(min=0)) + self.pos_proj(sincos(pos, scale=240.0)))\n        allowed = self.out_bags[bag.clamp(min=0)] & mask[..., None]\n        s = self.att_w(torch.tanh(self.att_v(h))).masked_fill(~allowed, float("-inf"))\n        if crops:\n            assert not self.bagsum\n            return s, h @ self.cls_w.T + self.cls_b\n        if self.bagsum:\n            out = self.cls_b.expand(b, -1)\n            for j in range(self.n_bags):\n                a = torch.softmax(s.masked_fill((bag != j)[..., None], float("-inf")), 1).nan_to_num(0.0)\n                out = out + (torch.einsum("bkn,bkd->bnd", a, h) * self.cls_w[:, j]).sum(-1)\n            return out\n        a = torch.softmax(s, 1).nan_to_num(0.0)\n        return (torch.einsum("bkn,bkd->bnd", a, h) * self.cls_w).sum(-1) + self.cls_b\n\n\ndef bag_names(cfg):\n    bags, planes = cfg["bags"].split(","), cfg.get("planes", "")\n    keep = lambda p: not planes or p in planes.split(",")\n    return [f"{b}:{p}" for b in bags for p in PLANES_OF[b] if keep(p)] if cfg.get("plane_bags") else bags\n\n\ndef out_bags(cfg, names):\n    outs = cfg["outs"].split(",")\n    ob = np.ones((len(outs), len(names)), bool)\n    for kv in [x for x in cfg.get("out_bag", "").split("+") if x]:      # oct17: per-output view routing (syn_train order)\n        o, bs = kv.split(":")\n        ob[outs.index(o)] = [b.split(":")[0] in bs.split("|") for b in names]\n    if cfg.get("bakers_post", True) and "bakers" in outs and "post" in cfg["bags"].split(","):\n        ob[outs.index("bakers")] = [b.split(":")[0] == "post" for b in names]\n    return ob\n\n\ndef series_box(p, b, j, px=PX):\n    pres = p[:, j] > P_MIN\n    if not pres.any():\n        return None\n    bb = (b[pres, j] * p[pres, j, None]).sum(0) / p[pres, j].sum() * px\n    return (bb[0] + bb[1]) / 2, (bb[2] + bb[3]) / 2\n\n\n@torch.no_grad()\ndef localise(loc, vol, plane, pos, dev):\n    n = len(vol)\n    small = np.stack([cv2.resize(a, (LOC_PX, LOC_PX), interpolation=cv2.INTER_AREA) for a in vol])\n    w = np.stack([small[np.clip(np.arange(n) + d, 0, n - 1)] for d in (-1, 0, 1)], 1)\n    p, b = loc(torch.from_numpy(w).to(dev), torch.from_numpy(aux_features(plane, pos, n)).to(dev))\n    return torch.sigmoid(p).float().cpu().numpy(), b.float().cpu().numpy()\n\n\ndef study_bags(series, locs, cfg):\n    """series: list of dicts vol, plane, type, pos; locs: per series (p, b) or None -> bag name -> windows (si, i, cy, cx, side_px,\n    type, plane, pos, p), with main/syn_data.py build_bags + finalize rules (crops in the side-corrected view: no mirroring)."""\n    bags, planes = cfg["bags"].split(","), cfg.get("planes", "")\n    names = bag_names(cfg)\n    out = {n: [] for n in names}\n    for si, (s, lb) in enumerate(zip(series, locs)):\n        n = len(s["vol"])\n        if n < 3 or lb is None or (planes and s["plane"] not in planes.split(",")):\n            continue\n        p, b = lb\n        for bag in bags:\n            if s["plane"] not in PLANES_OF[bag]:\n                continue\n            j = STRUCTS_V2.index(STRUCT[bag])\n            c = series_box(p, b, j)\n            if c is None:\n                continue\n            cy, cx = (PX / 2, PX / 2) if bag in ("knee", "knee140") else c\n            side_px = CROP_MM[bag] * cfg.get("crop_scale", 1.0) / MM_PER_PX\n            if cfg.get("clamp"):\n                lo, hi = min(side_px / 2, PX / 2), max(PX - side_px / 2, PX / 2)\n                cy, cx = float(np.clip(cy, lo, hi)), float(np.clip(cx, lo, hi))\n            keep = [i for i in range(1, n - 1) if p[i, j] > P_MIN]\n            if len(keep) > 3:\n                cut = int(round(cfg.get("trim", 0.1) * len(keep)))\n                keep = keep[cut:len(keep) - cut]\n            key = f"{bag}:{s[\'plane\']}" if cfg.get("plane_bags") else bag\n            out[key] += [(si, i, cy, cx, side_px, s["type"], PLANE_ID[s["plane"]], float(s["pos"][i]), float(p[i, j])) for i in keep]\n    if cfg.get("fs_only"):\n        for key, c in out.items():\n            out[key] = [w for w in c if ki.SERIES_TYPES[w[5]].endswith("_FS")] or c\n    for key in names:\n        if not out[key]:\n            donor = [x for k in names if k != key for x in out[k]]\n            out[key] = sorted(donor, key=lambda w: -w[-1])[:TOP_FALLBACK]\n    return out\n\n\ndef crop(img3, cy, cx, side, px):\n    h = int(round(side))\n    y0, x0 = int(round(cy - side / 2)), int(round(cx - side / 2))\n    out = np.zeros((3, h, h), np.uint8)\n    sy, sx = max(y0, 0), max(x0, 0)\n    ey, ex = min(y0 + h, img3.shape[1]), min(x0 + h, img3.shape[2])\n    if ey > sy and ex > sx:\n        out[:, sy - y0:ey - y0, sx - x0:ex - x0] = img3[:, sy:ey, sx:ex]\n    if h == px:\n        return out\n    return np.stack([cv2.resize(np.ascontiguousarray(c), (px, px), interpolation=cv2.INTER_AREA if h > px else cv2.INTER_LINEAR) for c in out])\n\n\ndef study_tensors(series, bags, names, m, px, keep=None):\n    """keep (oct18_probe_acl_b3g, member cfg \'keep_bags\'): crops only for these bags; bag ids keep their index in the FULL bag list (bag\n    embedding / routing as trained). Bag coordinates (study_bags) are still built for every bag, so the empty-bag donor rule is unchanged."""\n    keep = set(keep.split(",")) if keep else None\n    kt = m * (len(keep) if keep else len(names))\n    x = np.zeros((kt, 3, px, px), np.uint8)\n    mask, bag = np.zeros(kt, bool), np.full(kt, -1, np.int64)\n    typ, plane, pos = np.zeros(kt, np.int64), np.zeros(kt, np.int64), np.zeros(kt, np.float32)\n    k = 0\n    for j, name in enumerate(names):\n        if keep is not None and name not in keep:\n            continue\n        c = bags[name]\n        for i in np.unique(np.linspace(0, len(c) - 1, min(m, len(c))).round().astype(int)) if c else []:\n            si, sl, cy, cx, side, t, pl, ps, _ = c[i]\n            x[k] = crop(series[si]["vol"][[sl - 1, sl, sl + 1]], cy, cx, side, px)\n            mask[k], bag[k], typ[k], plane[k], pos[k] = True, j, t, pl, ps\n            k += 1\n    t = lambda a, dt: torch.from_numpy(np.ascontiguousarray(a[None])).to(dt)\n    return torch.from_numpy(x[None]), t(mask, torch.bool), t(typ, torch.long), t(plane, torch.long), t(pos, torch.float32), t(bag, torch.long)\n\n\ndef correct_sides(series, det, dev, thr=0.9):\n    """Basin\'s mirror detector on each series; flipped back like the training flip table. Returns the number flipped."""\n    n = 0\n    with torch.no_grad():\n        for s in series:\n            if len(s["vol"]) < 3:\n                continue\n            p = torch.sigmoid(det(torch.from_numpy(det_input(s["vol"], s["plane"]))[None].to(dev))).item()\n            if p > thr:\n                n += 1\n                if s["plane"] == "Sagittal":\n                    s["vol"], s["pos"] = s["vol"][::-1].copy(), -np.asarray(s["pos"])[::-1].copy()\n                else:\n                    s["vol"] = s["vol"][..., ::-1].copy()\n    return n\n\n\ndef load_experts(members, dev):\n    """members: {name: {"cfg": syn_train config, "files": [weight paths]}} -> {name: (models, cfg, names, mean, std)}."""\n    experts = {}\n    for name, e in members.items():\n        c = e["cfg"]\n        names = bag_names(c)\n        ms = []\n        for f in e["files"]:\n            m = SynNet(c["backbone"], out_bags(c, names), len(names), len(ki.SERIES_TYPES), c.get("hidden", 256), c.get("bagsum", False))\n            m.load_state_dict(torch.load(f, map_location="cpu"))\n            ms.append(m.to(dev).to(memory_format=torch.channels_last).eval())\n        cfgb = ms[0].backbone.pretrained_cfg\n        mean = torch.tensor(cfgb["mean"], device=dev).view(1, 3, 1, 1)\n        std = torch.tensor(cfgb["std"], device=dev).view(1, 3, 1, 1)\n        experts[name] = (ms, c, names, mean, std)\n    return experts\n\n\nCROP_KEYS = ("bags", "planes", "fs_only", "plane_bags", "trim", "crop_scale", "clamp", "k_eval", "px", "keep_bags")\n\n\ndef fuse_design(S, U, groups, gate=20):\n    """oct13_f5_win synovitis crop fusion (experiments/v2/oct13_deep_ensembling_test wfuse spec \'mean|pm|g20|maxmix\', w_synx.py\n    \'design\'): S, U (models, k) of one design\'s members (both backbones) on its shared crops; relevance = mean over backbones of the\n    backbone\'s mean attention prob; value = mean over backbones of the mean window prob; top-`gate` crops; pooled prob =\n    0.7 x relevance-weighted mean + 0.3 x max gated value -> logit."""\n    A = np.exp(S - S.max(1, keepdims=True))\n    A = A / A.sum(1, keepdims=True)\n    g = np.asarray(groups)\n    r = np.stack([A[g == k].mean(0) for k in sorted(set(groups))]).mean(0)\n    P = np.stack([(1 / (1 + np.exp(-U[g == k]))).mean(0) for k in sorted(set(groups))]).mean(0)\n    keep = np.zeros_like(r)\n    keep[np.argsort(-r)[:gate]] = 1.0\n    r = r * keep\n    r = r / r.sum()\n    p = np.clip(0.7 * (r * P).sum() + 0.3 * np.where(r > 0, P, 0).max(), 1e-6, 1 - 1e-6)\n    return float(np.log(p / (1 - p)))\n\n\n@torch.no_grad()\ndef expert_logits(experts, series, locs, dev, fuse=None):\n    """Crops are built once per crop configuration (the fold models of one design share them) and reused. fuse (config syn "fuse",\n    oct13_f5_win): the members\' outputs come from the per-crop path (same math) and out["fused@<design>"] = fuse_design of each\n    design\'s members (both backbones) on img_syn."""\n    out, cache = {}, {}\n    member_of = {n: (d, gi) for d, e in (fuse or {}).get("designs", {}).items() for gi, ns in enumerate(e.values()) for n in ns}\n    crops = {}\n    for name, (ms, c, names, mean, std) in experts.items():\n        key = tuple(str(c.get(k)) for k in CROP_KEYS)\n        if key not in cache:\n            bags = study_bags(series, locs, c)\n            cache[key] = tuple(t.to(dev) for t in study_tensors(series, bags, names, c["k_eval"], c["px"], c.get("keep_bags")))\n        x, mask, typ, plane, pos, bag = cache[key]\n        xf = x.flatten(0, 1).float() / 255\n        if c.get("mono"):                                 # centre slice in all three channels (main/data.py gpu_prep mono)\n            xf = xf[:, 1:2].expand(-1, 3, -1, -1)\n        xf = ((xf - mean) / std).contiguous(memory_format=torch.channels_last)\n        outs = c["outs"].split(",")\n        lg = []\n        for m in ms:\n            with torch.autocast(dev.type, dtype=AMP, enabled=dev.type == "cuda"):\n                if fuse:          # oct16_syn_cnx_probe: every member takes the per-crop path (0.961: all members are in a design)\n                    sc, uc = m(xf, mask, typ, plane, pos, bag, crops=True)\n                    sc, uc = sc.float()[0], uc.float()[0]\n                    lg.append((torch.softmax(sc, 0).nan_to_num(0.0) * uc).sum(0))\n                    if name in member_of:\n                        j = outs.index("img_syn")\n                        d, gi = member_of[name]\n                        crops.setdefault(d, []).append((gi, sc[:, j].cpu().numpy().astype(np.float64), uc[:, j].cpu().numpy().astype(np.float64)))\n                else:\n                    lg.append(m(xf, mask, typ, plane, pos, bag).float()[0])\n        o = torch.stack(lg).mean(0)\n        if not torch.isfinite(o).all():\n            raise FloatingPointError(f"non-finite expert {name} logit")\n        if "img_syn" in outs:\n            out[name] = float(o[outs.index("img_syn")])\n        if "img_eff" in outs:                             # oct13_f5_eff: the image-read Effusion head of the same forward pass\n            out[f"{name}@img_eff"] = float(o[outs.index("img_eff")])\n        for j, on in enumerate(outs):                     # oct17: every text output of a non-synovitis member (e.g. w5b_mcl@mcl)\n            if on not in ("img_syn", "img_eff"):\n                out[f"{name}@{on}"] = float(o[j])\n    for d, cs in crops.items():\n        z = fuse_design(np.stack([c[1] for c in cs]), np.stack([c[2] for c in cs]), [c[0] for c in cs], fuse["gate"])\n        out[f"fused@{d}"] = z if np.isfinite(z) else float("nan")\n    return out\n\n\nclass Studies(torch.utils.data.Dataset):\n    def __init__(self, root, studies, fs_of):\n        self.root, self.studies, self.fs_of = root, list(studies), fs_of\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        s = self.studies[i]\n        try:\n            series, _ = ki.study_series(f"{self.root}/{s}", self.fs_of)\n            return s, [{"vol": d["vol"], "plane": d["plane"], "type": int(d["type"]), "pos": np.asarray(d["pos"], np.float32)} for d in series], ""\n        except Exception as e:  # noqa: BLE001\n            return s, [], f"{type(e).__name__}: {e}"\n\n\ndef predict(root, studies, fs_of, members, loc_path, det_path, workers=4, fuse=None):\n    """-> {expert name: Series (study -> synovitis logit)}; failed studies are missing (STATS[\'failed\'])."""\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    t0 = time.time()\n    loc = Loc()\n    loc.load_state_dict(torch.load(loc_path, map_location="cpu"))\n    loc = loc.to(dev).to(memory_format=torch.channels_last).eval()\n    det = FlipNet(pretrained=False)\n    det.load_state_dict(torch.load(det_path, map_location="cpu"))\n    det = det.to(dev).eval()\n    experts = load_experts(members, dev)\n    dl = torch.utils.data.DataLoader(Studies(root, studies, fs_of), batch_size=None, num_workers=workers, collate_fn=lambda x: x,\n                                     prefetch_factor=2 if workers else None)\n    rows = collections.defaultdict(dict)\n    STATS.update(failed=[], flipped=0)\n    for sid, series, err in dl:\n        try:\n            if err or not series:\n                raise RuntimeError(err or "no readable series")\n            STATS["flipped"] += correct_sides(series, det, dev)\n            locs = [localise(loc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n            for k, v in expert_logits(experts, series, locs, dev, fuse).items():\n                rows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            STATS["failed"].append(sid)\n            print(f"syn_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n    STATS["wall_s"] = time.time() - t0\n    return {k: pd.Series(v, dtype=np.float64) for k, v in rows.items()}\n', "ensemble.py": '"""oct20_stack_all (Experts V3 10-19; USER GO via Master: the merged fold package for a 10-20 slot; OAI: no) = oct19_stack_fold\'s file + :\n- Contusion FS (cont_fs, LB-won 57388555): the bfs_fs bone member (fc \'nofuse\') spliced into Contusion after the bone block (its probe\'s code);\n- SWITCHES (config keys present only when ON; each = its probe\'s own code): frac_mfs (m_fs fsmain member -> Fracture splice, 57399358),\n  oa extra_sets {"pf": ...} + oa extra_align (PF Q20: PF from the fusion over ONLY oasep / oasep_nf, 57399364), bakers_allpw (Q25: the pw\n  Basin models join the Basin pass for window export only, the pw main models the main window pass; Baker\'s = their window fusion, 57399558),\n  lm_allpw (C184 G1, no probe of its own: LM\'s 3x3 base = the same pw generalists\' study logits - a pw main pass + the pw Basin models in the\n  Basin pass - before the meniscus recipe; the meniscus recipe is per column, so only LM changes);\n- stage-argument helpers basin_args / oa_cfg / fc_cfg / syn_members, shared with fast.py (the two-GPU runner) so both paths call each stage\n  with identical arguments. The post-hoc DEC (MCL Q6 re-fitted on the new Contusion, LM Q22) stays in the notebook.\noct19_stack_fold (Experts V3 10-18, Master: the LB-won column forms stacked on the 0.961 fold package; each block below is its\nprobe\'s own code, keyed by its config entry, on disjoint columns): MCL = Q21 (mcl_sd + mcl_member + mcl_member2; Q6 in the notebook),\nLOA = Q15 (oa loa_extra / loa_set / loa_extra_align), ACL = Q16 (acl_member + acl_member2), Synovitis = the ConvNeXt crop slots of\n57283240 (syn pipelines / fuse designs; eff_pipelines keeps Effusion on the effb0 crop experts). Every other column = 0.961.\noct18_probe_mcl_stack (Q21, USER U55 stack): = oct18_probe_mcl_sd2\'s file + config "mcl_member2" (b3g\'s area MCL spliced after the zoom member).\noct18_probe_mcl_sd2: + config "mcl_sd" (MCL = mean of the main SD / NFNet SD / deployed R50 logits, replacing the 3x3 MCL before the\nMCL zoom splice). oct17_probe_mcl_zoom: + config "mcl_member" (one extra crop member in the syn stage, spliced into the MCL column; see below).\n3-pipeline ensemble from raw DICOM (same file runs locally and on Kaggle):\n- main: knee_infer.py (effb0, experiments/v0/oct7_baseline) on its config.json models; probability = mean over its models, synovitis\n  logit mixed with effusion per model (config synovitis_effusion_w), exactly as the 0.940 package;\n- Basin-code pipelines: basin_predict.py on config.json "basin" = {pipeline: [checkpoint files in the package]};\n  pipeline logit = mean of its models\' logits (all Basin models share one DICOM pass); config "basin_detector" (optional): Basin\'s\n  mirror detector, which mirrors back series that the test-time side rule mis-sided (Basin models trained on corrected sides; the\n  main pipeline trained on its own DICOM sides, so it gets no detector).\nFinal probability = sigmoid(mean over pipelines of the pipeline logits): equal weight per pipeline (validated by the 3rd baseline:\nlogit mean ~ z-scored mean ~ prob mean within +-0.0003 OOF).\noct10_syn3x3: Synovitis comes only from image-head models (trained on the Codex image reads, experiments/v1/oct10_synovitis):\n- main: the 13-output models in config "models" (knee_infer\'s mixed mode: Synovitis from the 13-output models\' image head with\n  synovitis_ext_w = 1, the other 11 findings from the 12-output models; synovitis_effusion_w must be 0);\n- Basin pipelines: config "basin_syn" = {pipeline: [image-head checkpoints]} (basin_predict returns their image head as Synovitis);\nSynovitis = sigmoid(mean over the 3 pipelines of the pipeline image-head logits); the 11 other findings exactly as oct10_3pipe_3f.\noct11_oa probes (config "oa", experiments/v1/oct10_oa/probes.md): only the 3 OA columns change, to ONE model type (single model vs\nsingle model, no blending): mode "main" = the main pipeline\'s own OA logits (baseline B); mode "expert" = one compartment-crop expert\n(oa_stage.py, mean logit of its fold models); a study the expert cannot read falls back to the main pipeline\'s OA logits (logged).\noct12_syn3pipe (config "syn"): Synovitis from the synovitis crop experts of experiments/v1/oct11_synovitis_crops (syn_stage.py:\nlocaliser v2 crops, Codex read target) on the 3 pipelines\' encoders: per pipeline the mean logit of its experts (2 designs x\nfolds 0-2), Synovitis = sigmoid(mean over the 3 pipelines); a study the stage cannot read keeps the image-head Synovitis (logged).\noct12_syn_g (config "syn" image_heads_w = w): Synovitis = sigmoid(w x image-head logit (mean over the pipelines in "basin_syn" + main)\n+ (1 - w) x crop-expert logit); w = 0 is oct12_syn3pipe.\noct12_men_3pipe: config "men" (make_men_recipe.py, folds 0-2 OOF): Medial / Lateral Meniscus = the pipelines\' meniscus logits blended\nwith the OA experts\' meniscus heads (same expert pass): o = (1 - w) b + w z(e), z = e rescaled to b\'s OOF mean / sd.\noct12_fc_3pipe (config "fc", experiments/v1/oct11_fracture_contusion): Fracture / Contusion = fc_stage.apply_recipe(pipelines\' F / C\nlogits, F / C expert members\' logits, fc recipe) = (1 - w) b + w z(e), chosen on folds 0-2 OOF; members: bone experts (localiser v3\ncrops); a study the stage cannot read keeps the pipelines\' F / C (logged).\noct12_global_best: oct12_syn_g + the "men" block of oct12_men_3pipe + the "fc" block of oct12_fc_3pipe (disjoint columns).\noct13_f5_win: oct13_f5_eff + config syn "fuse": the synovitis crop-expert part from crop-level fusion per design (syn_stage.fuse_design,\nmean|pm|g20|maxmix) + config oa "fuse": the OA experts\' term (OA and meniscus recipes) from their crop-level fusion\n(oa_stage, fc_stage.fuse_crops, geo|pm|g10|wmean) + config "win": Effusion / Baker\'s from the cross-backbone window fusion of the 3x3 (win_stage.py: main\nmodels on every window + Basin per-image export, geo|pm|g20|wmean), before the expert recipes; + config fc "fuse": Fracture / Contusion from the crop-level fusion of the bone experts (fc_stage.fuse_crops:\nevery member\'s fold models over the shared crops, top-20 relevance gate, prob mean) in place of the recipe\'s member z-mean.\noct13_f5_eff (experiments/v2/oct13_deep_ensembling_test): the existing fold 3-4 checkpoints of main / NFNet / R50 / bone b_v3, b_lr3\n(config lists only) + config "eff": Effusion blended with the synovitis crop experts\' image-read Effusion head (oa_stage.apply_recipe\nform, no de-coupling)."""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\nimport knee_infer as ki  # noqa: E402\nimport basin_predict as bp  # noqa: E402\nimport oa_stage  # noqa: E402\nimport syn_stage  # noqa: E402\nimport fc_stage  # noqa: E402\nimport win_stage  # noqa: E402\n\nFINDINGS = ki.FINDINGS\nLOA = "Lateral OA"\nSTATS = {}\n\n\ndef logit(p):\n    p = np.clip(np.asarray(p, np.float64), 1e-7, 1 - 1e-7)\n    return np.log(p / (1 - p))\n\n\ndef basin_args(cfg, pkg):\n    """-> (ckpts, detector, win kwargs or None): the ONE Basin pass (deployed + basin_syn + mcl_sd NFNet SD + bakers_allpw pw models for window\n    export only), as ensemble.predict and fast.py call bp.predict."""\n    det = str(Path(pkg) / cfg["basin_detector"]) if cfg.get("basin_detector") else None\n    msd, bk, win, lp = cfg.get("mcl_sd"), cfg.get("bakers_allpw"), cfg.get("win"), cfg.get("lm_allpw")\n    ckpts = [str(Path(pkg) / c) for cs in list(cfg["basin"].values()) + list(cfg["basin_syn"].values()) + ([msd["nfnet"]] if msd else []) for c in cs]\n    bkck = {p: [str(Path(pkg) / c) for c in cs] for p, cs in bk["basin"].items()} if bk else {}\n    ckpts += sum(bkck.values(), [])\n    if lp:                 # SWITCH lm_allpw: the pw Basin models\' study logits (one run per checkpoint when bakers_allpw is also ON)\n        ckpts += [c for c in (str(Path(pkg) / x) for cs in lp["basin"].values() for x in cs) if c not in ckpts]\n    wck = {p: [str(Path(pkg) / c) for c in cs] for p, cs in win["basin"].items()} if win else {}\n    wkw = {"ckpts": sum(wck.values(), []) + sum(bkck.values(), []), "cols": [FINDINGS.index(c) for c in win["cols"]]} if win else None\n    return ckpts, det, wkw\n\n\ndef oa_cfg(cfg):\n    return cfg["oa"]\n\n\ndef fc_cfg(cfg):\n    return cfg["fc"]\n\n\ndef syn_members(cfg, pkg):\n    members = json.load(open(Path(pkg) / cfg["syn"]["members"]))\n    for m in members.values():\n        m["files"] = [str(Path(pkg) / f) for f in m["files"]]\n    return members\n\n\ndef predict(root, studies, series, pkg, threads=None, workers=4):\n    """series: the host series csv (StudyInstanceUID, SeriesInstanceUID, Fat_Suppression). Returns (final probabilities, parts =\n    probabilities {pipeline or Basin checkpoint file: DataFrame}, failed main studies, per-study main load seconds)."""\n    cfg = json.load(open(Path(pkg) / "config.json"))\n    series = series[series.StudyInstanceUID.isin(studies)].assign(Fat_Suppression=lambda d: d.Fat_Suppression.fillna(0).astype(int))\n    fs_of = dict(zip(series.SeriesInstanceUID, series.Fat_Suppression))\n    t = time.time()   # Basin first: its DataLoader forks workers before knee_infer starts its thread pools\n    msd = cfg.get("mcl_sd")    # oct18_probe_mcl_sd2: the NFNet SD fold models run in the same Basin pass (MCL only)\n    win = cfg.get("win")   # oct13_f5_win: cross-backbone window fusion for win["cols"] (win_stage.py)\n    wcols = [FINDINGS.index(c) for c in win["cols"]] if win else None\n    wck = {p: [str(Path(pkg) / c) for c in cs] for p, cs in win["basin"].items()} if win else {}\n    bk = cfg.get("bakers_allpw")   # SWITCH Q25: pw basin models for window export only (appended; never in the pipeline means)\n    bkck = {p: [str(Path(pkg) / c) for c in cs] for p, cs in bk["basin"].items()} if bk else {}\n    ckpts, det, wkw = basin_args(cfg, pkg)\n    z = bp.predict(series, root, ckpts, workers=workers, detector=det, win=wkw)\n    STATS["basin"] = {"wall_s": time.time() - t, **bp.STATS}\n    torch.cuda.empty_cache()   # knee_infer captures its CUDA graphs with nothing else on the GPU\n    t = time.time()\n    p_main, failed, secs = ki.predict(root, studies, fs_of, pkg, threads)\n    STATS["main"] = {"wall_s": time.time() - t, **ki.STATS}\n    torch._C._host_emptyCache()   # oct13_f5_win: knee_infer\'s cached pinned study buffers (Kaggle: 12.7 GB of the 30 GB after 117 studies)\n    if msd:                       # oct18_probe_mcl_sd2: a second main pass with the 5 main SD fold models (own weights dir; MCL only)\n        torch.cuda.empty_cache()\n        t = time.time()\n        p_sd, failed_sd, _ = ki.predict(root, studies, fs_of, str(Path(pkg) / msd["main_dir"]), threads)\n        STATS["main_sd"] = {"wall_s": time.time() - t, "failed": len(failed_sd)}\n        torch._C._host_emptyCache()\n    lp = cfg.get("lm_allpw")   # SWITCH lm_allpw (C184 G1): a main pass with the 5 pw main fold models (own weights dir; LM only)\n    if lp:\n        torch.cuda.empty_cache()\n        t = time.time()\n        p_pw, failed_pw, _ = ki.predict(root, studies, fs_of, str(Path(pkg) / lp["main_dir"]), threads)\n        STATS["main_pw"] = {"wall_s": time.time() - t, "failed": len(failed_pw)}\n        torch._C._host_emptyCache()\n    assert cfg["synovitis_ext_w"] == 1.0 and cfg["synovitis_effusion_w"] == 0.0, "Synovitis = the image heads alone"\n    zs = {"main": pd.DataFrame(logit(p_main.loc[studies, FINDINGS]), index=pd.Index(studies, name="StudyInstanceUID"), columns=FINDINGS)}\n\n    def pipe_logit(p, cs):\n        zp = sum(z[str(Path(pkg) / c)] for c in cs) / len(cs)\n        miss = [s for s in studies if s not in zp.index]   # study without rows in the series csv: cohort mean logit (logged)\n        if miss:\n            print(f"basin {p}: {len(miss)} studies not in the series csv -> cohort mean: {miss[:10]}", flush=True)\n        return zp.reindex(studies).fillna(zp.mean())\n    for p, cs in cfg["basin"].items():\n        zs[p] = pipe_logit(p, cs)\n    zsyn = {"main": zs["main"]["Synovitis"]} | {p: pipe_logit(p, cs)["Synovitis"] for p, cs in cfg["basin_syn"].items()}\n    final_z = sum(zs.values()) / len(zs)\n    zs_sd = {}\n    if msd:                       # MCL = (main SD + NFNet SD + deployed R50) / 3; a study the SD main pass failed keeps the deployed main\'s MCL\n        z_ms = pd.Series(logit(p_sd.loc[studies, ["MCL"]]).ravel(), index=pd.Index(studies, name="StudyInstanceUID"))\n        z_ms[[s for s in studies if s in failed_sd]] = zs["main"].loc[[s for s in studies if s in failed_sd], "MCL"]\n        z_nfs = pipe_logit("nfnet_sd", msd["nfnet"])["MCL"]\n        zs_sd = {"mcl_sd_main": z_ms.to_frame("MCL"), "mcl_sd_nfnet": z_nfs.to_frame("MCL")}   # parts (checks only), added after final_z\n        final_z["MCL"] = (z_ms + z_nfs + zs[msd["r50"]]["MCL"]) / 3\n        print(f"mcl_sd: MCL = (main SD + NFNet SD + {msd[\'r50\']}) / 3; SD main failed {len(failed_sd)}", flush=True)\n    if lp:                        # LM base = (pw main + pw NFNet + pw R50) / 3; a study the pw main pass failed keeps the deployed main\'s LM\n        c = lp["col"]\n        z_mp = pd.Series(logit(p_pw.loc[studies, [c]]).ravel(), index=pd.Index(studies, name="StudyInstanceUID"))\n        z_mp[[s for s in studies if s in failed_pw]] = zs["main"].loc[[s for s in studies if s in failed_pw], c]\n        lpz = {"lm_pw_main": z_mp.to_frame(c)} | {f"lm_pw_{p}": pipe_logit(p, cs)[[c]] for p, cs in lp["basin"].items()}\n        zs_sd |= lpz                                  # parts (checks only)\n        final_z[c] = sum(v[c] for v in lpz.values()) / len(lpz)\n        print(f"lm_allpw: {c} base = mean of {list(lpz)}; pw main failed {len(failed_pw)}", flush=True)\n    if win:\n        torch.cuda.empty_cache()\n        t = time.time()\n        n0 = len(win["main"])\n        mw = win_stage.main_windows(root, studies, fs_of, str(pkg), win["main"] + (bk["main"] if bk else []), cfg, wcols)\n        fz, fz2, miss = {}, {}, []\n        for sid in studies:\n            if sid in mw and all(c in bp.WIN.get(sid, {}) for cs in wck.values() for c in cs):\n                m_ = mw[sid]\n                fz[sid] = win_stage.fuse(m_ | {"s": m_["s"][:n0], "u": m_["u"][:n0]}, {p: [bp.WIN[sid][c] for c in cs] for p, cs in wck.items()}, win["gate"])\n                if bk and all(c in bp.WIN.get(sid, {}) for cs in bkck.values() for c in cs):     # SWITCH Q25 (its probe\'s code)\n                    fz2[sid] = win_stage.fuse(m_ | {"s": m_["s"][n0:], "u": m_["u"][n0:]}, {p: [bp.WIN[sid][c] for c in cs] for p, cs in bkck.items()},\n                                              win["gate"])\n            else:\n                miss.append(sid)\n        if miss:\n            print(f"win: {len(miss)} studies without window outputs -> study-level 3x3 for {win[\'cols\']}: {miss[:10]}", flush=True)\n        fz = pd.DataFrame.from_dict(fz, orient="index", columns=win["cols"]).reindex(studies)\n        zs["win_fused"] = fz.copy()\n        final_z[win["cols"]] = fz.fillna(final_z[win["cols"]])\n        if bk:                     # Q25: Baker\'s from the pw fusion; a study without pw windows keeps the deployed fusion (logged)\n            assert set(bk["cols"]) <= set(win["cols"]), bk["cols"]\n            assert all(v["s"].shape[0] == n0 + len(bk["main"]) for v in mw.values()), "bakers_allpw main models missing from the window pass"\n            fz2 = pd.DataFrame.from_dict(fz2, orient="index", columns=win["cols"]).reindex(studies)[bk["cols"]]\n            zs["win_fused_bk"] = fz2.copy()\n            final_z[bk["cols"]] = fz2.fillna(final_z[bk["cols"]])\n        STATS["win"] = {"wall_s": time.time() - t, "missing": len(miss), "bakers_allpw": int(fz2.notna().all(axis=1).sum()) if bk else 0}\n    oa = cfg.get("oa")\n    if oa:\n        main_oa = zs["main"][oa_stage.OA]\n        if oa["mode"] == "main":\n            final_z[oa_stage.OA] = main_oa\n        else:\n            t = time.time()\n            m = oa_stage.predict(root, studies, fs_of, pkg, oa_cfg(cfg), workers=workers, threads=threads)\n            fused = m.pop("fused", None)\n            fx = m.pop("fused_loa_extra", None)\n            fxs = {k: m.pop(f"fused_{k}") for k in oa.get("extra_sets", {})}   # SWITCH PF Q20: extra fusions (never in the deployed mean)\n            dep = {k: v for k, v in m.items() if k not in set(oa.get("loa_extra", []))}   # oct16_loa_bundle: extras only enter LOA\'s fusion\n            e = (sum(dep.values()) / len(dep)).reindex(studies).astype(np.float64)\n            ef = None\n            if fused is not None:   # oct13_f5_win: crop-level fusion of the OA experts, aligned to e\'s OOF distribution, replaces e\n                al = oa["fuse"]["align"]\n                ef = pd.DataFrame({f: (fused[f] - al[f]["mu_f"]) / al[f]["sd_f"] * al[f]["sd_e"] + al[f]["mu_e"] for f in al}).reindex(studies)\n                zs["oa_fused"] = ef.copy()\n                e = ef[e.columns].fillna(e)\n            miss = e.index[e.isna().any(axis=1)]\n            if "recipe" in oa:      # oct11_oa_3pipe: pipelines\' OA (final_z) blended with the experts\' mean + de-coupling\n                if len(miss):\n                    print(f"oa: {len(miss)} studies without expert logits -> pipelines\' OA in the recipe: {list(miss)[:10]}", flush=True)\n                b_oa = final_z[oa_stage.OA].copy()\n                final_z[oa_stage.OA] = oa_stage.apply_recipe(b_oa, e, oa["recipe"])\n                if fx is not None:    # oct16_loa_bundle (10-16 12:03): Lateral OA from the deployed + extra members\' fusion (own align,\n                    # oa "loa_extra_align"); its de-coupling reads the deployed MOA / PF; the MOA / PF outputs (their de-couplings read\n                    # LOA) and meniscus stay as deployed (output-level splice, as submissions/oct16_probe_loa_splice)\n                    a = oa["loa_extra_align"]\n                    ex = e.copy()\n                    ex[LOA] = ((fx[LOA] - a["mu_f"]) / a["sd_f"] * a["sd_e"] + a["mu_e"]).reindex(studies).fillna(e[LOA])\n                    zs["oa_fused_loa_extra"] = ex[[LOA]].copy()\n                    final_z[LOA] = oa_stage.apply_recipe(b_oa, ex, oa["recipe"])[LOA]\n                for k, fk in fxs.items():   # SWITCH PF Q20 (oct18_probe_pf_allup\'s splice): column extra_align[k]["col"] from its own fusion\n                    a, C = oa["extra_align"][k], oa["extra_align"][k]["col"]\n                    ex2 = e.copy()\n                    ex2[C] = ((fk[C] - a["mu_f"]) / a["sd_f"] * a["sd_e"] + a["mu_e"]).reindex(studies).fillna(e[C])\n                    zs[f"oa_fused_{k}"] = ex2[[C]].copy()\n                    final_z[C] = oa_stage.apply_recipe(b_oa, ex2, oa["recipe"])[C]\n            else:\n                if len(miss):\n                    print(f"oa: {len(miss)} studies without expert logits -> main pipeline OA: {list(miss)[:10]}", flush=True)\n                final_z[oa_stage.OA] = e[oa_stage.OA].fillna(main_oa)\n            men = cfg.get("men")\n            if men:                 # oct12_men_3pipe: the same experts\' meniscus heads (no extra model pass) -> meniscus recipe\n                em = (sum(m[k] for k in men["experts"]) / len(men["experts"])).reindex(studies).astype(np.float64)[oa_stage.MEN]\n                if ef is not None:\n                    em = ef[oa_stage.MEN].fillna(em)\n                miss = em.index[em.isna().any(axis=1)]\n                if len(miss):\n                    print(f"men: {len(miss)} studies without expert logits -> pipelines\' meniscus: {list(miss)[:10]}", flush=True)\n                final_z[oa_stage.MEN] = oa_stage.apply_recipe(final_z[oa_stage.MEN], em, men)\n            zs["oa_expert"] = final_z.copy()\n            zs |= {f"oa_{k}": v.reindex(studies).astype(np.float64) for k, v in m.items()}   # each expert\'s own OA logits (checks only)\n            STATS["oa"] = {"wall_s": time.time() - t, **{k: v for k, v in oa_stage.STATS.items() if k != "failed"},\n                           "failed": len(oa_stage.STATS.get("failed", []))}\n    fc = cfg.get("fc")\n    if fc:\n        torch.cuda.empty_cache()\n        t = time.time()\n        m = {k: v.reindex(studies).astype(np.float64) for k, v in fc_stage.predict_pkg(root, studies, fs_of, pkg, fc_cfg(cfg), workers=workers).items()}\n        miss = [s for s in studies if any(np.isnan(v.loc[s]).any() for v in m.values())]\n        if miss:\n            print(f"fc: {len(miss)} studies without expert logits -> pipelines\' F / C: {miss[:10]}", flush=True)\n        fc_base = final_z[fc_stage.FC].copy()\n        final_z[fc_stage.FC] = fc_stage.apply_recipe(fc_base, m, fc["recipe"])\n        if fc.get("fuse"):      # oct13_f5_win: crop-level fusion of all bone members replaces the recipe\'s member z-mean\n            zf = fc_stage.apply_recipe_fused(fc_base, m["fused"], fc["recipe"], fc["fuse"])\n            zs["fc_member_recipe"] = final_z[fc_stage.FC].copy()           # checks only\n            final_z[fc_stage.FC] = zf.fillna(final_z[fc_stage.FC])\n        cf = cfg.get("cont_fs")    # oct17_probe_cont_fs (LB-won): Contusion = (1 - w) x package logit + w x z(FS-only bone member); label-free align\n        if cf:\n            zc = m[cf["name"]]["Contusion"]\n            zs["cont_fs"] = zc.to_frame("Contusion")\n            zl = (zc - cf["mu_m"]) / cf["sd_m"] * cf["sd_d"] + cf["mu_d"]\n            final_z["Contusion"] = ((1 - cf["w"]) * final_z["Contusion"] + cf["w"] * zl).fillna(final_z["Contusion"])   # no member logit -> deployed\n            print(f"cont_fs: {cf[\'name\']} spliced into Contusion at w {cf[\'w\']} ({int(zc.notna().sum())} / {len(zc)} studies)", flush=True)\n        fm = cfg.get("frac_mfs")   # SWITCH oct19_probe_frac_mfs: Fracture = (1 - w) x package logit + w x z(m_fs Fracture logit); label-free align\n        if fm:\n            zf_ = m[fm["name"]]["Fracture"]\n            zs["frac_mfs"] = zf_.to_frame("Fracture")\n            zl = (zf_ - fm["mu_m"]) / fm["sd_m"] * fm["sd_d"] + fm["mu_d"]\n            final_z["Fracture"] = ((1 - fm["w"]) * final_z["Fracture"] + fm["w"] * zl).fillna(final_z["Fracture"])   # no member logit -> deployed\n            print(f"frac_mfs: {fm[\'name\']} spliced into Fracture at w {fm[\'w\']} ({int(zf_.notna().sum())} / {len(zf_)} studies)", flush=True)\n        zs |= {f"fc_{k}": v for k, v in m.items()}   # each member\'s own F / C logits (checks only)\n        STATS["fc"] = {"wall_s": time.time() - t, **{k: v for k, v in fc_stage.STATS.items() if k != "failed"},\n                       "failed": len(fc_stage.STATS.get("failed", []))}\n    syn, synm, zih_parts = cfg.get("syn"), {}, dict(zsyn)\n    zsyn_final = sum(zsyn.values()) / len(zsyn)\n    if syn:\n        torch.cuda.empty_cache()   # the crop stage loads 18 models next to knee_infer\'s CUDA-graph pools (16 GB T4)\n        t = time.time()\n        members = syn_members(cfg, pkg)\n        zm = pd.DataFrame(syn_stage.predict(root, studies, fs_of, members, str(Path(pkg) / syn["loc"]), str(Path(pkg) / syn["detector"]),\n                                            workers=workers, fuse=syn.get("fuse"))).reindex(studies).astype(np.float64)\n        ze = sum(zm[names].mean(1) for names in syn["pipelines"].values()) / len(syn["pipelines"])\n        if syn.get("fuse"):     # oct13_f5_win: crop-level fusion per design (both backbones), mean over designs, replaces the crop part\n            zf = zm[[f"fused@{d}" for d in syn["fuse"]["designs"]]].mean(1)\n            zs["syn_fused"] = zf.to_frame("Synovitis")\n            ze = zf.fillna(ze)\n        miss = ze.index[ze.isna()]\n        if len(miss):\n            print(f"syn: {len(miss)} studies without crop-expert logits -> image-head Synovitis: {list(miss)[:10]}", flush=True)\n        w = syn.get("image_heads_w", 0.0)\n        zsyn = {"image_heads": zsyn_final, "crop_experts": ze.fillna(zsyn_final)}\n        zsyn_final = w * zsyn["image_heads"] + (1 - w) * zsyn["crop_experts"]\n        eff = cfg.get("eff")\n        if eff:                 # oct13_f5_eff: Effusion = (1 - w) b + w z(e), e = the crop experts\' image-read Effusion (same pass)\n            effp = syn.get("eff_pipelines", syn["pipelines"])   # oct16_syn_cnx_probe: Effusion keeps the effb0 crop experts\n            ee = sum(zm[[f"{n}@img_eff" for n in names]].mean(1) for names in effp.values()) / len(effp)\n            miss = ee.index[ee.isna()]\n            if len(miss):\n                print(f"eff: {len(miss)} studies without crop-expert logits -> pipelines\' Effusion: {list(miss)[:10]}", flush=True)\n            zs["eff_expert"] = ee.to_frame("Effusion")\n            final_z[["Effusion"]] = oa_stage.apply_recipe(final_z[["Effusion"]], ee.to_frame("Effusion"), eff)\n        mm = cfg.get("mcl_member")   # oct17_probe_mcl_zoom: MCL = (1 - w) x package logit + w x z(member MCL logit); label-free align\n        if mm:\n            zmc = zm[f"{mm[\'name\']}@mcl"]\n            zs["mcl_member"] = zmc.to_frame("MCL")\n            zl = (zmc - mm["mu_m"]) / mm["sd_m"] * mm["sd_d"] + mm["mu_d"]\n            final_z["MCL"] = ((1 - mm["w"]) * final_z["MCL"] + mm["w"] * zl).fillna(final_z["MCL"])   # no member logit -> deployed MCL\n            print(f"mcl_member: {mm[\'name\']} spliced into MCL at w {mm[\'w\']} ({int(zmc.notna().sum())} / {len(zmc)} studies)", flush=True)\n        m2 = cfg.get("mcl_member2")  # oct18_probe_mcl_stack: MCL\'\' = (1 - w) x MCL\' (the SD-2 + zoom form above) + w x z(b3g area MCL); z aligned to MCL\' OOF\n        if m2:\n            zm2 = zm[f"{m2[\'name\']}@mcl"]\n            zs["mcl_member2"] = zm2.to_frame("MCL")\n            zl2 = (zm2 - m2["mu_m"]) / m2["sd_m"] * m2["sd_d"] + m2["mu_d"]\n            final_z["MCL"] = ((1 - m2["w"]) * final_z["MCL"] + m2["w"] * zl2).fillna(final_z["MCL"])   # no member logit -> the SD-2 + zoom form\n            print(f"mcl_member2: {m2[\'name\']} spliced into MCL at w {m2[\'w\']} ({int(zm2.notna().sum())} / {len(zm2)} studies)", flush=True)\n        am = cfg.get("acl_member")   # oct18_probe_acl_b3g (Q16): ACL = (1 - w) x package logit + w x z(member ACL logit); label-free align\n        if am:\n            zac = zm[f"{am[\'name\']}@acl"]\n            zs["acl_member"] = zac.to_frame("ACL")\n            zla = (zac - am["mu_m"]) / am["sd_m"] * am["sd_d"] + am["mu_d"]\n            final_z["ACL"] = ((1 - am["w"]) * final_z["ACL"] + am["w"] * zla).fillna(final_z["ACL"])   # no member logit -> deployed ACL\n            print(f"acl_member: {am[\'name\']} spliced into ACL at w {am[\'w\']} ({int(zac.notna().sum())} / {len(zac)} studies)", flush=True)\n        a2 = cfg.get("acl_member2")  # oct18_probe_acl_b3g (Q16): ACL\'\' = (1 - w) x ACL\' (the Z1 form above) + w x z(b3g area ACL); z aligned to ACL\' OOF\n        if a2:\n            za2 = zm[f"{a2[\'name\']}@acl"]\n            zs["acl_member2"] = za2.to_frame("ACL")\n            zla2 = (za2 - a2["mu_m"]) / a2["sd_m"] * a2["sd_d"] + a2["mu_d"]\n            final_z["ACL"] = ((1 - a2["w"]) * final_z["ACL"] + a2["w"] * zla2).fillna(final_z["ACL"])   # no member logit -> the Z1 form\n            print(f"acl_member2: {a2[\'name\']} spliced into ACL at w {a2[\'w\']} ({int(za2.notna().sum())} / {len(za2)} studies)", flush=True)\n        synm = {f"synm_{k}": 1 / (1 + np.exp(-v)) for k, v in zm.items()}            # each expert fold model (checks only)\n        STATS["syn"] = {"wall_s": time.time() - t, "flipped": syn_stage.STATS.get("flipped"), "failed": len(syn_stage.STATS.get("failed", []))}\n    final = 1 / (1 + np.exp(-final_z))\n    final["Synovitis"] = 1 / (1 + np.exp(-zsyn_final))\n    parts = {k: 1 / (1 + np.exp(-v)) for k, v in (zs | zs_sd).items()} | {f"syn_{k}": 1 / (1 + np.exp(-v)) for k, v in zsyn.items()} | \\\n            {f"syn_ih_{k}": 1 / (1 + np.exp(-v)) for k, v in zih_parts.items()} | \\\n            {c: 1 / (1 + np.exp(-z[str(Path(pkg) / c)])) for cs in list(cfg["basin"].values()) + list(cfg["basin_syn"].values()) for c in cs} | synm\n    return final, parts, failed, secs\n', "oa_stage.py": '"""OA stage of the oct11_oa packages: compartment-crop OA experts (experiments/v1/oct10_oa) blended into the 3-pipeline OA logits.\nCopied (flat, no experiment imports) from experiments/v1/oct10_oa: scripts/oa_expert_infer.py (verified test-time path, corr 0.9999 vs\nOOF), main/oa_train.py OANet + encoder, main/model.py sincos, main/oa_data.py constants, scripts/loc_train.py Loc + aux_features.\n\nPer study:\n  1. knee_infer.study_series: the CPU path, byte-identical to the training cache fov160_px400 (uint8 (n, 400, 400), 0.4 mm/px);\n  2. Basin\'s mirror detector (basin_flipdet, p > 0.9) per series: the experts were trained in the side-corrected view (flip table\n     data/derived/series_side_final.csv); a series called mirrored is flipped back like the table did (sagittal: stack reversed,\n     pos -> -pos reversed; coronal / axial: columns mirrored);\n  3. localiser on every slice (128 px 3-slice windows; v2 with 16 structures for loc_version v2cor) -> series boxes -> compartment bags (medial / lateral: all planes, lateral\n     coronal / axial crops mirrored; pf: sagittal + axial) -> k_eval windows per bag, crops of CROP_MM resized to the expert\'s px;\n  4. each expert = mean logit over its fold models (OUTS = Medial OA, Lateral OA, PF OA, Medial / Lateral Meniscus: the experts\'\n     auxiliary meniscus heads come from the same forward pass, oct12_men_3pipe); mono experts (RadImageNet R50) read\n     the centre slice in all three channels, as in training.\nMembers (config "oa" -> "members"): kind "expert" (the steps above) or kind "main" (main-pipeline effb0 models run through\nknee_infer.predict on a temporary weights dir with their own config: the same-setting control, trained the normal way on all 12 labels).\nBlend (ensemble.py): per OA finding F, with OOF statistics of the experiment (pooled folds 0-2, scripts in make_oa_config.py):\n  z_m = (m - mu_m) * sd_b / sd_m + mu_b per member;  o = (1 - w) * b + w * mean_m(z_m);  out = o_F - b_F * (c0 + c . other two o).\nA study a member cannot read keeps the pipeline blend for OA (logged)."""\nimport json\nimport math\nimport os\nimport tempfile\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport timm\nimport torch\nimport torch.nn as nn\n\nimport fc_stage\nimport knee_infer as ki\nfrom basin_flipdet import FlipNet, det_input\n\nOA = ["Medial OA", "Lateral OA", "PF OA"]\nMEN = ["Medial Meniscus", "Lateral Meniscus"]\nOUTS = OA + MEN          # the experts\' 5 outputs (oa_train.py OUTS): oct12_men_3pipe also uses their meniscus heads\nOUT_BAG = [0, 1, 2, 0, 1]\nBAGS = ["medial", "lateral", "pf"]\nSTRUCTS = ["comp_medial", "comp_lateral", "pf_joint", "patella", "joint"]\nSTRUCT = {"medial": "comp_medial", "lateral": "comp_lateral", "pf": "pf_joint"}\nSTRUCTS_V2 = STRUCTS + ["med_tib_cart", "lat_tib_cart", "pat_cart", "fem_cart", "med_men", "lat_men", "notch", "mcl", "suprapatellar",\n                        "hoffa", "popliteal"]\nSTRUCT_V2 = {"medial": "med_tib_cart", "lateral": "lat_tib_cart", "pf": "pat_cart"}\nPLANES = ["Sagittal", "Coronal", "Axial"]\nPLANE_ID = {p: i for i, p in enumerate(PLANES)}\nPLANES_OF = {"medial": ("Sagittal", "Coronal", "Axial"), "lateral": ("Sagittal", "Coronal", "Axial"), "pf": ("Sagittal", "Axial")}\nCROP_MM = {("medial", "Sagittal"): 72, ("medial", "Coronal"): 56, ("medial", "Axial"): 56,\n           ("lateral", "Sagittal"): 72, ("lateral", "Coronal"): 56, ("lateral", "Axial"): 56,\n           ("pf", "Sagittal"): 64, ("pf", "Axial"): 64}\nP_MIN, TRIM, TOP_FALLBACK, LOC_PX, MM_PER_PX = 0.5, 0.15, 3, 128, 0.4\nSTATS = {}\nAMP = getattr(torch, os.environ.get("OA_AMP_DTYPE", "float16"))   # T4: fp16; local checks may set bfloat16 = the experiment\n\n\ndef sincos(pos, n_freq=8, scale=60.0):\n    f = torch.exp2(torch.arange(n_freq, device=pos.device, dtype=pos.dtype)) * (2 * math.pi / scale)\n    a = pos[..., None] * f\n    return torch.cat([a.sin(), a.cos()], -1)\n\n\nclass Loc(nn.Module):\n    def __init__(self, arch="mobilenetv3_large_100", structs=STRUCTS):\n        super().__init__()\n        self.n_s = len(structs)\n        self.backbone = timm.create_model(arch, pretrained=False, num_classes=0)\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.head = nn.Sequential(nn.Linear(d + 5, 256), nn.ReLU(), nn.Linear(256, self.n_s * 5))\n        c = self.backbone.pretrained_cfg\n        self.register_buffer("mean", torch.tensor(c["mean"]).view(1, 3, 1, 1) * 255)\n        self.register_buffer("std", torch.tensor(c["std"]).view(1, 3, 1, 1) * 255)\n\n    def forward(self, x, aux):\n        x = ((x.float() - self.mean) / self.std).contiguous(memory_format=torch.channels_last)\n        o = self.head(torch.cat([self.backbone(x), aux], 1)).view(-1, self.n_s, 5)\n        return o[..., 0], torch.sigmoid(o[..., 1:])\n\n\ndef aux_features(plane, pos, n):\n    p = np.zeros((n, 3), np.float32)\n    p[:, PLANE_ID[plane]] = 1\n    return np.concatenate([p, (np.asarray(pos, np.float32) / 60)[:, None], (np.arange(n) / max(n - 1, 1)).astype(np.float32)[:, None]], 1)\n\n\nclass OANet(nn.Module):\n    """main/oa_train.py OANet (attention / lse / topk / hybrid pooling over each output\'s own bag)."""\n    def __init__(self, backbone, n_types, hidden=256, split=False, bml=False, lse_r=0.0, topk=0, hybrid=False, pl_head=False):\n        super().__init__()\n        assert not (split and pl_head)\n        self.lse_r, self.topk, self.hybrid = lse_r, topk, hybrid\n        self.backbone = timm.create_model("resnet50" if backbone == "radimagenet_resnet50" else backbone, pretrained=False, num_classes=0)\n        if backbone == "radimagenet_resnet50":   # oa_train.encoder: timm resnet50 with RadImageNet weights, ImageNet mean / std\n            self.backbone.pretrained_cfg = dict(self.backbone.pretrained_cfg, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.type_emb, self.plane_emb, self.pos_proj = nn.Embedding(n_types, d), nn.Embedding(3, d), nn.Linear(16, d)\n        self.norm = nn.LayerNorm(d)\n        out_bag = OUT_BAG + ([0, 1, 2] if bml else []) + ([0, 1, 2] if pl_head else [])   # pl_head (oct16_loa_bundle): 3 \'T:\' OA heads last\n        self.head_of = ([0, 0, 1, 2, 2] + ([3, 3, 4] if bml else [])) if split else list(range(len(out_bag)))\n        n_head = max(self.head_of) + 1\n        self.att_v, self.att_w = nn.Linear(d, hidden), nn.Linear(hidden, n_head)\n        self.cls_w = nn.Parameter(torch.zeros(n_head, d))\n        self.cls_w2 = nn.Parameter(torch.zeros(n_head, d)) if hybrid else None\n        self.cls_b = nn.Parameter(torch.zeros(len(out_bag)))\n        self.register_buffer("out_bag", torch.tensor(out_bag), persistent=False)\n\n    def inst(self, h, allowed, w):\n        return torch.einsum("bkd,nd->bkn", h, w[self.head_of]).masked_fill(~allowed, float("-inf"))\n\n    def lme(self, s, allowed):\n        return (torch.logsumexp(self.lse_r * s, 1) - torch.log(allowed.sum(1).clamp(min=1).float())) / self.lse_r\n\n    def forward(self, x, mask, typ, plane, pos, bag, crops=False):\n        """crops=True (oct13_f5_win; plain attention head): per-crop attention logits s (-inf outside an output\'s bag) and logits u\n        (b, k, n_out); the study logit is exactly sum_k softmax_k(s) u."""\n        b, k = mask.shape\n        f = self.backbone(x).float().view(b, k, -1)\n        h = self.norm(f + self.type_emb(typ) + self.plane_emb(plane) + self.pos_proj(sincos(pos, scale=240.0)))\n        allowed = (bag[..., None] == self.out_bag) & mask[..., None]\n        if crops:\n            assert not self.topk and not self.lse_r and not self.hybrid\n            s = self.att_w(torch.tanh(self.att_v(h)))[..., self.head_of].masked_fill(~allowed, float("-inf"))\n            return s, h @ self.cls_w[self.head_of].T + self.cls_b\n        if self.topk:\n            v = self.inst(h, allowed, self.cls_w).topk(min(self.topk, k), dim=1).values\n            fin = torch.isfinite(v)\n            return v.where(fin, torch.zeros_like(v)).sum(1) / fin.sum(1).clamp(min=1) + self.cls_b\n        if self.lse_r and not self.hybrid:\n            return self.lme(self.inst(h, allowed, self.cls_w), allowed) + self.cls_b\n        s = self.att_w(torch.tanh(self.att_v(h)))[..., self.head_of].masked_fill(~allowed, float("-inf"))\n        att = (torch.einsum("bkn,bkd->bnd", torch.softmax(s, 1), h) * self.cls_w[self.head_of]).sum(-1)\n        if self.hybrid:\n            return 0.5 * (att + self.lme(self.inst(h, allowed, self.cls_w2), allowed)) + self.cls_b\n        return att + self.cls_b\n\n\ndef series_box(p, b, j, px=400):\n    pres = p[:, j] > 0.5\n    if not pres.any():\n        return None\n    bb = (b[pres, j] * p[pres, j, None]).sum(0) / p[pres, j].sum() * px\n    return (bb[0] + bb[1]) / 2, (bb[2] + bb[3]) / 2\n\n\n@torch.no_grad()\ndef localise(loc, vol, plane, pos, dev):\n    n = len(vol)\n    small = np.stack([cv2.resize(a, (LOC_PX, LOC_PX), interpolation=cv2.INTER_AREA) for a in vol])\n    w = np.stack([small[np.clip(np.arange(n) + d, 0, n - 1)] for d in (-1, 0, 1)], 1)\n    p, b = loc(torch.from_numpy(w).to(dev), torch.from_numpy(aux_features(plane, pos, n)).to(dev))\n    return torch.sigmoid(p).float().cpu().numpy(), b.float().cpu().numpy()\n\n\ndef study_bags(series, loc, dev, structs=STRUCTS, cor_v2=False):\n    """series: list of dicts vol, plane, type (int), pos -> bag -> windows (series index, slice, mirror, cy, cx, side_px, type, plane, pos, p).\n    cor_v2 (loc_version v2cor, localiser v2): coronal TF bags select slices and centre crops by tibial-cartilage presence."""\n    out = {b: [] for b in BAGS}\n    for si, s in enumerate(series):\n        n = len(s["vol"])\n        if n < 3:\n            continue\n        p, b = localise(loc, s["vol"], s["plane"], s["pos"], dev)\n        for bag in BAGS:\n            if s["plane"] not in PLANES_OF[bag]:\n                continue\n            j = structs.index(STRUCT_V2[bag] if cor_v2 and s["plane"] == "Coronal" and bag != "pf" else STRUCT[bag])\n            c = series_box(p, b, j)\n            if c is None:\n                continue\n            keep = [i for i in range(1, n - 1) if p[i, j] > P_MIN]\n            if len(keep) > 3:\n                cut = int(round(TRIM * len(keep)))\n                keep = keep[cut:len(keep) - cut]\n            mirror = bag == "lateral" and s["plane"] != "Sagittal"\n            side_px = CROP_MM[bag, s["plane"]] / MM_PER_PX\n            out[bag] += [(si, i, mirror, c[0], c[1], side_px, s["type"], PLANE_ID[s["plane"]], float(s["pos"][i]), float(p[i, j]))\n                         for i in keep]\n    for bag in BAGS:\n        if not out[bag]:\n            donor = [x for k in BAGS if k != bag for x in out[k]]\n            out[bag] = sorted(donor, key=lambda w: -w[-1])[:TOP_FALLBACK]\n    return out\n\n\ndef crop(img3, cy, cx, side, mirror, px):\n    h = int(round(side))\n    y0, x0 = int(round(cy - side / 2)), int(round(cx - side / 2))\n    out = np.zeros((3, h, h), np.uint8)\n    sy, sx = max(y0, 0), max(x0, 0)\n    ey, ex = min(y0 + h, img3.shape[1]), min(x0 + h, img3.shape[2])\n    if ey > sy and ex > sx:\n        out[:, sy - y0:ey - y0, sx - x0:ex - x0] = img3[:, sy:ey, sx:ex]\n    if mirror:\n        out = out[:, :, ::-1]\n    return np.stack([cv2.resize(np.ascontiguousarray(c), (px, px), interpolation=cv2.INTER_LINEAR) for c in out])\n\n\ndef study_tensors(series, bags, m, px):\n    x = np.zeros((len(BAGS), m, 3, px, px), np.uint8)\n    mask = np.zeros((len(BAGS), m), bool)\n    meta = np.zeros((len(BAGS), m, 3), np.float32)\n    for j, bag in enumerate(BAGS):\n        c = bags[bag]\n        sel = np.unique(np.linspace(0, len(c) - 1, min(m, len(c))).round().astype(int))\n        for k, w in enumerate(c[i] for i in sel):\n            si, i, mirror, cy, cx, side, typ, plane, pos, _ = w\n            x[j, k] = crop(series[si]["vol"][[i - 1, i, i + 1]], cy, cx, side, mirror, px)\n            mask[j, k] = True\n            meta[j, k] = (typ, plane, pos)\n    bag = np.broadcast_to(np.arange(len(BAGS))[:, None], (len(BAGS), m))\n    t = lambda a, dt: torch.from_numpy(np.ascontiguousarray(a.reshape(1, -1))).to(dt)\n    return (torch.from_numpy(x.reshape(1, -1, 3, px, px)), t(mask, torch.bool), t(meta[..., 0], torch.long), t(meta[..., 1], torch.long),\n            t(meta[..., 2], torch.float32), t(np.where(mask, bag, -1), torch.long))\n\n\ndef correct_sides(series, det, dev, thr=0.9):\n    """Basin\'s mirror detector on each series; flipped back like the training flip table. Returns the number flipped."""\n    n = 0\n    with torch.no_grad():\n        for s in series:\n            if len(s["vol"]) < 3:\n                continue\n            p = torch.sigmoid(det(torch.from_numpy(det_input(s["vol"], s["plane"]))[None].to(dev))).item()\n            if p > thr:\n                n += 1\n                if s["plane"] == "Sagittal":\n                    s["vol"], s["pos"] = s["vol"][::-1].copy(), -np.asarray(s["pos"])[::-1].copy()\n                else:\n                    s["vol"] = s["vol"][..., ::-1].copy()\n    return n\n\n\ndef load_experts(pkg, ocfg, dev):\n    experts = {}\n    for name, e in ocfg["members"].items():\n        if e["kind"] != "expert":\n            continue\n        c = e["cfg"]\n        ms = []\n        for f in e["files"]:\n            m = OANet(c["backbone"], ocfg["n_types"], c.get("hidden", 256), c.get("split", False), c.get("bml_weight", 0) > 0,\n                      c.get("lse_r", 0.0), c.get("topk", 0), c.get("hybrid", False), c.get("pl_head", False))\n            sd = torch.load(f"{pkg}/{f}", map_location="cpu")\n            missing, unexpected = m.load_state_dict(sd, strict=False)\n            assert not missing and not [u for u in unexpected if not u.startswith(("rel_proj", "cls_w2"))], (f, missing, unexpected)\n            ms.append(m.to(dev).eval())\n        mean = torch.tensor(ms[0].backbone.pretrained_cfg["mean"], device=dev).view(1, 3, 1, 1)\n        std = torch.tensor(ms[0].backbone.pretrained_cfg["std"], device=dev).view(1, 3, 1, 1)\n        # output columns read as OUTS: the 5 main heads, or (head \'T\', oct16_loa_bundle) the 3 teacher-target OA heads + the meniscus heads\n        n_out = len(ms[0].out_bag)\n        cols = [n_out - 3 + j for j in range(3)] + [3, 4] if e.get("head") == "T" else list(range(len(OUTS)))\n        assert e.get("head") != "T" or c.get("pl_head"), f"{name}: head \'T\' needs a pl_head model"\n        experts[name] = (ms, c["k_eval"], c["px"], mean, std, c.get("mono", False), cols)\n    return experts\n\n\n@torch.no_grad()\ndef expert_logits(experts, series, bags, dev, fuse=None, loa_extra=frozenset(), loa_set=frozenset(), extra_sets=None):\n    """fuse (config oa "fuse", oct13_f5_win): also out["fused"] = crop-level fusion (fc_stage.fuse_crops) of every expert\'s fold\n    models over the shared crops (experts must share k_eval / px / mono); outputs with an empty bag stay NaN (-> deployed e).\n    loa_extra (config oa "loa_extra", oct16_loa_bundle): \'fused\' stays the fusion over the deployed members (extras excluded: an empty\n    set = oct13_f5_win exactly); out["fused_loa_extra"] = the fusion over deployed + extra members, read for Lateral OA only\n    (ensemble.py splices it at the output level, 10-16 12:03).\n    extra_sets (config oa "extra_sets", oct20_stack_all: PF Q20): {key: member names} -> out[f"fused_{key}"] = the fusion over ONLY those\n    members (as loa_set), e.g. {"pf": [oa_oasep, oa_oasep_nf]}; None / {} = the stack exactly."""\n    out, cache, crops = {}, {}, []\n    for gi, (name, (ms, m_win, px, mean, std, mono, cols)) in enumerate(experts.items()):\n        if (m_win, px) not in cache:              # experts with one crop configuration share the crops (built once)\n            cache[(m_win, px)] = tuple(t.to(dev) for t in study_tensors(series, bags, m_win, px))\n        x, mask, typ, plane, pos, bag = cache[(m_win, px)]\n        xf = x.flatten(0, 1).float() / 255\n        if mono:                                  # oa_train mono (data.gpu_prep): the centre slice in all three channels (R50)\n            xf = xf[:, 1:2].expand(-1, 3, -1, -1)\n        xf = ((xf - mean) / std).contiguous(memory_format=torch.channels_last)\n        lg = []\n        for m in ms:\n            with torch.autocast("cuda", dtype=AMP, enabled=dev.type == "cuda"):\n                if fuse:\n                    assert not mono and len(cache) == 1, "OA crop fusion needs one shared crop configuration"\n                    sc, uc = m(xf, mask, typ, plane, pos, bag, crops=True)\n                    sc, uc = sc.float()[0][:, cols], uc.float()[0][:, cols]\n                    crops.append((gi, sc, uc, name))\n                    lg.append((torch.softmax(sc, 0) * uc).sum(0))\n                else:\n                    lg.append(m(xf, mask, typ, plane, pos, bag).float()[0, cols])\n        v = torch.stack(lg).mean(0).cpu().numpy()\n        if not np.isfinite(v).all():\n            raise FloatingPointError(f"non-finite expert {name} logits")\n        out[name] = v\n    if fuse:\n        dep = [c for c in crops if c[3] not in loa_extra]\n        z = fc_stage.fuse_crops(torch.stack([c[1] for c in dep]), torch.stack([c[2] for c in dep]), [c[0] for c in dep],\n                                fuse["gate"]).cpu().numpy()\n        out["fused"] = np.where(np.isfinite(z), z, np.nan)\n        if loa_extra:                             # fusion over deployed + extra members (Lateral OA read only)\n            # oct18_probe_loa_allup (U50): loa_set = the ONLY members of the LOA fusion (the upgraded effb0 / NFNet / silL groups replace the\n            # deployed ones for LOA); empty = deployed + extras (the silL probe form)\n            la = [c for c in crops if c[3] in loa_set] if loa_set else crops\n            assert la, "loa_set matches no expert"\n            za = fc_stage.fuse_crops(torch.stack([c[1] for c in la]), torch.stack([c[2] for c in la]), [c[0] for c in la],\n                                     fuse["gate"]).cpu().numpy()\n            out["fused_loa_extra"] = np.where(np.isfinite(za), za, np.nan)\n        for key, names in (extra_sets or {}).items():  # oct20_stack_all: one more fusion per extra set (no extra model pass)\n            xs = [c for c in crops if c[3] in set(names)]\n            assert xs, f"extra set {key} matches no expert"\n            zx = fc_stage.fuse_crops(torch.stack([c[1] for c in xs]), torch.stack([c[2] for c in xs]), [c[0] for c in xs],\n                                     fuse["gate"]).cpu().numpy()\n            out[f"fused_{key}"] = np.where(np.isfinite(zx), zx, np.nan)\n    return out\n\n\nclass Studies(torch.utils.data.Dataset):\n    def __init__(self, root, studies, fs_of):\n        self.root, self.studies, self.fs_of = root, list(studies), fs_of\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        s = self.studies[i]\n        try:\n            series, skipped = ki.study_series(f"{self.root}/{s}", self.fs_of)\n            return s, [{"vol": d["vol"], "plane": d["plane"], "type": int(d["type"]), "pos": np.asarray(d["pos"], np.float32)}\n                       for d in series], ""\n        except Exception as e:  # noqa: BLE001\n            return s, [], f"{type(e).__name__}: {e}"\n\n\ndef main_member(root, studies, fs_of, pkg, member, threads=None):\n    """Main-pipeline models (knee_infer) as an OA member: probabilities averaged over its fold models (knee_infer\'s rule) -> logits."""\n    with tempfile.TemporaryDirectory() as tmp:\n        for f in member["files"]:\n            os.symlink(os.path.abspath(f"{pkg}/{f}"), f"{tmp}/{f}")\n        json.dump(member["cfg"] | {"weights": member["files"]}, open(f"{tmp}/config.json", "w"))\n        p, failed, _ = ki.predict(root, studies, fs_of, tmp, threads)\n    p = p.drop(index=list(failed))\n    lg = np.log(np.clip(p[OA].to_numpy(np.float64), 1e-7, 1 - 1e-7) / np.clip(1 - p[OA].to_numpy(np.float64), 1e-7, 1))\n    return pd.DataFrame(lg, index=p.index, columns=OA)\n\n\ndef apply_recipe(b, e, rec):\n    """OA logits of the oct11_oa_3pipe recipe (config "oa" -> "recipe", chosen by make_oa_recipe.py on folds 0-2 OOF).\n    b: the pipelines\' mean OA logits, e: the experts\' mean OA logits (DataFrames with the OA columns, same index; a NaN row in e =\n    no expert read the study: it keeps b). Per finding F: z = (e - mu_e) * sd_b / sd_e + mu_b (OOF statistics), o = (1 - w) * b + w * z,\n    then de-coupling out_F = o_F - k_F * (c0 + c . o of the other two OA findings) for the F in rec["dec"] ([k_F, c0, c1, c2])."""\n    o = b.copy()\n    for f in rec["stats"]:       # OA recipe: the 3 OA columns; oct12_men_3pipe meniscus recipe: the 2 meniscus columns (no "dec")\n        s = rec["stats"][f]\n        z = (e[f] - s["mu_e"]) * s["sd_b"] / s["sd_e"] + s["mu_b"]\n        o[f] = (1 - rec["w"]) * b[f] + rec["w"] * z.fillna(b[f])\n    out = o.copy()\n    for f, (k, c0, *c) in rec["dec"].items():\n        out[f] = o[f] - k * (c0 + o[[g for g in OA if g != f]].to_numpy() @ np.asarray(c))\n    return out\n\n\ndef predict(root, studies, fs_of, pkg, ocfg, workers=4, threads=None):\n    """-> {member name: DataFrame of member logits (experts: studies x OUTS = 3 OA + 2 meniscus heads; main members: OA)};\n    failed studies are missing (logged)."""\n    out = {name: main_member(root, studies, fs_of, pkg, m, threads) for name, m in ocfg["members"].items() if m["kind"] == "main"}\n    if not any(m["kind"] == "expert" for m in ocfg["members"].values()):\n        return out\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    t0 = time.time()\n    cor_v2 = ocfg.get("loc_version", "v1") == "v2cor"\n    structs = STRUCTS_V2 if cor_v2 else STRUCTS\n    loc = Loc(structs=structs)\n    loc.load_state_dict(torch.load(f"{pkg}/{ocfg[\'loc\']}", map_location="cpu"))\n    loc = loc.to(dev).to(memory_format=torch.channels_last).eval()\n    det = FlipNet(pretrained=False)\n    det.load_state_dict(torch.load(f"{pkg}/{ocfg[\'detector\']}", map_location="cpu"))\n    det = det.to(dev).eval()\n    experts = load_experts(pkg, ocfg, dev)\n    dl = torch.utils.data.DataLoader(Studies(root, studies, fs_of), batch_size=None, num_workers=workers, collate_fn=lambda x: x,\n                                     prefetch_factor=2 if workers else None)   # identity collate: keep numpy volumes\n    rows = {k: {} for k in list(experts) + (["fused"] if ocfg.get("fuse") else []) + (["fused_loa_extra"] if ocfg.get("loa_extra") else [])\n            + [f"fused_{k}" for k in ocfg.get("extra_sets", {})]}\n    STATS.update(failed=[], flipped=0)\n    for sid, series, err in dl:\n        try:\n            if err or not series:\n                raise RuntimeError(err or "no readable series")\n            STATS["flipped"] += correct_sides(series, det, dev)\n            bags = study_bags(series, loc, dev, structs, cor_v2)\n            for k, v in expert_logits(experts, series, bags, dev, ocfg.get("fuse"), set(ocfg.get("loa_extra", [])),\n                                      set(ocfg.get("loa_set", [])), ocfg.get("extra_sets")).items():\n                rows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            STATS["failed"].append(sid)\n            print(f"oa_stage: study {sid} failed ({type(e).__name__}: {e}); OA keeps the pipeline blend", flush=True)\n    STATS["wall_s"] = time.time() - t0\n    return out | {k: pd.DataFrame.from_dict(v, orient="index", columns=OUTS).astype(np.float64) for k, v in rows.items()}\n\n\ndef blend(z, oa, ocfg):\n    """z: pipeline-mean logits (studies x FINDINGS); oa: predict() output -> z with the OA columns replaced (C3 / C4 recipe)."""\n    out = z.copy()\n    b = z[OA]\n    have = b.index.intersection(*[d.index for d in oa.values()]) if oa else b.index[:0]\n    zs = []\n    for name, d in oa.items():\n        st = ocfg["members"][name]\n        e = d.loc[have, OA]\n        zs.append(pd.DataFrame({f: (e[f] - st["mu"][f]) * ocfg["blend_sd"][f] / st["sd"][f] + ocfg["blend_mu"][f] for f in OA}))\n    o = b.copy()\n    o.loc[have] = (1 - ocfg["w"]) * b.loc[have] + ocfg["w"] * sum(zs) / len(zs)\n    for f in OA:\n        dc = ocfg["decouple"][f]\n        others = [g for g in OA if g != f]\n        out[f] = o[f] - dc["b"] * (dc["intercept"] + sum(c * o[g] for c, g in zip(dc["coef"], others)))\n    return out\n', "fc_stage.py": '"""Test-time path of the fracture / contusion bone experts (main/bone_data.py + main/bone_train.py BoneNet), adapted from\nexperiments/v1/oct11_synovitis_crops/deploy/syn_stage.py. Flat: no experiment imports, so it can be copied into a Kaggle package.\n\nPer study:\n  1. knee_infer.study_series: the CPU path, byte-identical to the training cache fov160_px400 (uint8 (n, 400, 400), 0.4 mm/px);\n  2. Basin\'s mirror detector (p > 0.9) per series, flipped back like the training flip table (sagittal: stack reversed, pos -> -pos;\n     coronal / axial: columns mirrored): the experts were trained in the side-corrected view;\n  3. localiser v3 (18 structures: v2 + femur / tibia) on every slice -> series boxes (presence-weighted mean over slices with\n     p > 0.5) -> bone bags with the training rules of each expert\'s config (bags, planes, series, trim, crop_scale);\n  4. each expert = mean Fracture / Contusion logits over its fold models (k_eval windows per bag, evenly spread).\nMembers of kind "fsmain" (pipeline-1 main models trained with series_keep=fs, e.g. runs/m_fs) are scored before step 2, as the main\nmodels are (no side correction): fat-suppressed series only, knee_infer.windows (k_eval evenly spread (series, centre) pairs in the\n5-95% span = data.StudyDataset eval windows), KneeNet, mean Fracture / Contusion logits over the fold models.\npredict() -> {member name: DataFrame (study x Fracture, Contusion logits)}; studies that fail are missing (logged).\nPackage use (submissions/oct12_fc_3pipe, reusable in a combined package): config.json "fc" = {"members": {name: {kind, files, cfg}},\n"loc", "detector", "recipe"}; ensemble.py calls predict_pkg() and apply_recipe() (blend with the pipelines\' F / C logits). Source:\nexperiments/v1/oct11_fracture_contusion/deploy/bone_stage.py (verified vs the experiment on gold-58) + the two package functions.\nNeeds knee_infer.py, basin_flipdet.py, model.py of this toolchain (the same files the OA stage uses)."""\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nimport knee_infer as ki\nfrom basin_flipdet import FlipNet, det_input\nfrom model import KneeNet\n\nSTRUCTS_V3 = ["comp_medial", "comp_lateral", "pf_joint", "patella", "joint", "med_tib_cart", "lat_tib_cart", "pat_cart", "fem_cart", "med_men",\n              "lat_men", "notch", "mcl", "suprapatellar", "hoffa", "popliteal", "femur", "tibia"]\nSTRUCT = {"femur": "femur", "tibia": "tibia", "patella": "patella", "knee": "joint"}\nLOC_BAG = {"c_lfc": "femur", "c_mfc": "femur", "c_fem": "femur", "c_ltp": "tibia", "c_mtp": "tibia", "c_tib": "tibia", "c_fib": "tibia",\n           "c_pat": "patella", "f_fem": "femur", "f_tib": "tibia", "f_fib": "tibia", "f_pat": "patella"}\nFEM_H, TIB_H, MARGIN, TIB_XM, MIN_MM, MAX_MM = 72.0, 64.0, 8.0, 16.0, 48.0, 120.0\nPLANE_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}\nP_MIN, TOP_FALLBACK, LOC_PX, MM_PER_PX, PX = 0.5, 3, 128, 0.4, 400\nAMP = getattr(torch, os.environ.get("BONE_AMP_DTYPE", "float16"))\nFINDINGS_FC = {"Fracture": 11, "Contusion": 10}                      # column of each finding in the 12 main outputs (common.paths order)   # T4: fp16; local checks may set bfloat16 = the experiment\nFC = ["Fracture", "Contusion"]\nSTATS = {}\n\n\ndef sincos(pos, n_freq=8, scale=60.0):\n    f = torch.exp2(torch.arange(n_freq, device=pos.device, dtype=pos.dtype)) * (2 * math.pi / scale)\n    a = pos[..., None] * f\n    return torch.cat([a.sin(), a.cos()], -1)\n\n\nclass Loc(nn.Module):\n    """scripts/loc_train.py Loc (MobileNetV3, 3-slice 128 px windows -> presence logit + box per structure)."""\n    def __init__(self, arch="mobilenetv3_large_100", structs=STRUCTS_V3):\n        super().__init__()\n        self.n_s = len(structs)\n        self.backbone = timm.create_model(arch, pretrained=False, num_classes=0)\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.head = nn.Sequential(nn.Linear(d + 5, 256), nn.ReLU(), nn.Linear(256, self.n_s * 5))\n        c = self.backbone.pretrained_cfg\n        self.register_buffer("mean", torch.tensor(c["mean"]).view(1, 3, 1, 1) * 255)\n        self.register_buffer("std", torch.tensor(c["std"]).view(1, 3, 1, 1) * 255)\n\n    def forward(self, x, aux):\n        x = ((x.float() - self.mean) / self.std).contiguous(memory_format=torch.channels_last)\n        o = self.head(torch.cat([self.backbone(x), aux], 1)).view(-1, self.n_s, 5)\n        return o[..., 0], torch.sigmoid(o[..., 1:])\n\n\ndef aux_features(plane, pos, n):\n    p = np.zeros((n, 3), np.float32)\n    p[:, PLANE_ID[plane]] = 1\n    return np.concatenate([p, (np.asarray(pos, np.float32) / 60)[:, None], (np.arange(n) / max(n - 1, 1)).astype(np.float32)[:, None]], 1)\n\n\nclass BoneNet(nn.Module):\n    """main/bone_train.py BoneNet (per-output attention pooling over the allowed bags; out_fs outputs read FS windows only)."""\n    def __init__(self, backbone, out_bags, n_bags, n_types, out_fs, fs_type, hidden=256, bagsum=False, bb_pool="avg"):\n        super().__init__()\n        self.bagsum, self.n_bags = bagsum, n_bags\n        self.backbone = timm.create_model("resnet50" if backbone == "radimagenet_resnet50" else backbone, pretrained=False, num_classes=0,\n                                          global_pool=bb_pool)\n        if backbone == "radimagenet_resnet50":   # bone_train.encoder: timm resnet50 with RadImageNet weights, ImageNet mean / std\n            self.backbone.pretrained_cfg = dict(self.backbone.pretrained_cfg, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))\n        with torch.no_grad():\n            d = self.backbone(torch.zeros(1, 3, 64, 64)).shape[1]\n        self.type_emb, self.plane_emb, self.bag_emb, self.pos_proj = nn.Embedding(n_types, d), nn.Embedding(3, d), nn.Embedding(n_bags, d), nn.Linear(16, d)\n        self.norm = nn.LayerNorm(d)\n        n_out = out_bags.shape[0]\n        self.att_v, self.att_w = nn.Linear(d, hidden), nn.Linear(hidden, n_out)\n        self.cls_w = nn.Parameter(torch.zeros(n_out, n_bags, d) if bagsum else torch.zeros(n_out, d))\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        self.register_buffer("out_bags", torch.as_tensor(out_bags).T.contiguous(), persistent=False)\n        self.register_buffer("out_fs", torch.as_tensor(out_fs), persistent=False)\n        self.register_buffer("fs_type", torch.as_tensor(fs_type), persistent=False)\n\n    def forward(self, x, mask, typ, plane, pos, bag, crops=False):\n        """crops=True (oct13_f5_win, no bagsum): per-crop attention logits s and logits u (b, k, n_out); the study logit is exactly\n        sum_k softmax_k(s) u."""\n        b, k = mask.shape\n        f = self.backbone(x).float().view(b, k, -1)\n        h = self.norm(f + self.type_emb(typ) + self.plane_emb(plane) + self.bag_emb(bag.clamp(min=0)) + self.pos_proj(sincos(pos, scale=240.0)))\n        allowed = self.out_bags[bag.clamp(min=0)] & mask[..., None]\n        if self.out_fs.any():\n            fs = allowed & self.fs_type[typ][..., None]\n            keep = ~self.out_fs | ~fs.any(1)\n            allowed = torch.where(keep[:, None, :], allowed, fs)\n        s = self.att_w(torch.tanh(self.att_v(h))).masked_fill(~allowed, float("-inf"))\n        if self.bagsum:\n            out = self.cls_b.expand(b, -1)\n            for j in range(self.n_bags):\n                a = torch.softmax(s.masked_fill((bag != j)[..., None], float("-inf")), 1).nan_to_num(0.0)\n                out = out + (torch.einsum("bkn,bkd->bnd", a, h) * self.cls_w[:, j]).sum(-1)\n            return out\n        if crops:\n            return s, h @ self.cls_w.T + self.cls_b\n        a = torch.softmax(s, 1).nan_to_num(0.0)\n        return (torch.einsum("bkn,bkd->bnd", a, h) * self.cls_w).sum(-1) + self.cls_b\n\n\ndef out_masks(cfg, names):\n    outs = cfg["outs"].split(",")\n    ob = np.ones((len(outs), len(names)), bool)\n    for j, o in enumerate(outs):\n        if cfg.get("loc_restrict", True) and o in LOC_BAG and LOC_BAG[o] in names:\n            ob[j] = [b == LOC_BAG[o] for b in names]\n    fs_outs = cfg.get("fs_outs", "").split(",")\n    return ob, np.array([o in fs_outs for o in outs])\n\n\ndef crop_box(bag, plane, b, crop_scale, pat_margin=MARGIN):\n    """bone_data.crop_box: series box (y0, y1, x0, x1) in cache px -> (cy, cx, side_px)."""\n    y0, y1, x0, x1 = b\n    m = (pat_margin if bag == "patella" else MARGIN) / MM_PER_PX\n    xm = (TIB_XM if bag == "tibia" else pat_margin if bag == "patella" else MARGIN) / MM_PER_PX\n    if plane != "Axial" and bag == "femur":\n        y0 = max(y0, y1 - FEM_H / MM_PER_PX)\n    elif plane != "Axial" and bag == "tibia":\n        y1 = min(y1, y0 + TIB_H / MM_PER_PX)\n    side = float(np.clip(max(y1 - y0 + 2 * m, x1 - x0 + 2 * xm), MIN_MM / MM_PER_PX, MAX_MM / MM_PER_PX)) * crop_scale\n    return (y0 + y1) / 2, (x0 + x1) / 2, side\n\n\ndef series_box(p, b, j, px=PX):\n    pres = p[:, j] > P_MIN\n    if not pres.any():\n        return None\n    return (b[pres, j] * p[pres, j, None]).sum(0) / p[pres, j].sum() * px\n\n\n@torch.no_grad()\ndef localise(loc, vol, plane, pos, dev):\n    n = len(vol)\n    small = np.stack([cv2.resize(a, (LOC_PX, LOC_PX), interpolation=cv2.INTER_AREA) for a in vol])\n    w = np.stack([small[np.clip(np.arange(n) + d, 0, n - 1)] for d in (-1, 0, 1)], 1)\n    p, b = loc(torch.from_numpy(w).to(dev), torch.from_numpy(aux_features(plane, pos, n)).to(dev))\n    return torch.sigmoid(p).float().cpu().numpy(), b.float().cpu().numpy()\n\n\ndef study_bags(series, locs, cfg):\n    """series: list of dicts vol, plane, type, pos; locs: per series (p, b) or None -> bag name -> windows (si, i, cy, cx, side_px,\n    type, plane, pos, p), with main/bone_data.py build_bags + finalize rules (crops in the side-corrected view: no mirroring)."""\n    names, planes = cfg["bags"].split(","), cfg.get("planes", "")\n    out = {n: [] for n in names}\n    for si, (s, lb) in enumerate(zip(series, locs)):\n        n = len(s["vol"])\n        if n < 3 or lb is None or (planes and s["plane"] not in planes.split(",")):\n            continue\n        p, b = lb\n        for bag in names:\n            j = STRUCTS_V3.index(STRUCT[bag])\n            box = series_box(p, b, j)\n            if box is None:\n                continue\n            cy, cx, side_px = (PX / 2, PX / 2, 160 / MM_PER_PX) if bag == "knee" else crop_box(bag, s["plane"], box, cfg.get("crop_scale", 1.0), cfg.get("pat_margin", MARGIN))\n            keep = [i for i in range(1, n - 1) if p[i, j] > P_MIN]\n            if len(keep) > 3:\n                cut = int(round(cfg.get("trim", 0.05) * len(keep)))\n                keep = keep[cut:len(keep) - cut]\n            out[bag] += [(si, i, cy, cx, side_px, s["type"], PLANE_ID[s["plane"]], float(s["pos"][i]), float(p[i, j])) for i in keep]\n    if cfg.get("series", "all") != "all":\n        want = cfg["series"] == "fs"\n        for key, c in out.items():\n            out[key] = [w for w in c if ki.SERIES_TYPES[w[5]].endswith("_FS") == want] or c\n    for key in names:\n        if not out[key]:\n            donor = [x for k in names if k != key for x in out[k]]\n            out[key] = sorted(donor, key=lambda w: -w[-1])[:TOP_FALLBACK]\n    return out\n\n\ndef crop(img3, cy, cx, side, px):\n    h = int(round(side))\n    y0, x0 = int(round(cy - side / 2)), int(round(cx - side / 2))\n    out = np.zeros((3, h, h), np.uint8)\n    sy, sx = max(y0, 0), max(x0, 0)\n    ey, ex = min(y0 + h, img3.shape[1]), min(x0 + h, img3.shape[2])\n    if ey > sy and ex > sx:\n        out[:, sy - y0:ey - y0, sx - x0:ex - x0] = img3[:, sy:ey, sx:ex]\n    if h == px:\n        return out\n    return np.stack([cv2.resize(np.ascontiguousarray(c), (px, px), interpolation=cv2.INTER_AREA if h > px else cv2.INTER_LINEAR) for c in out])\n\n\ndef study_tensors(series, bags, names, m, px):\n    kt = m * len(names)\n    x = np.zeros((kt, 3, px, px), np.uint8)\n    mask, bag = np.zeros(kt, bool), np.full(kt, -1, np.int64)\n    typ, plane, pos = np.zeros(kt, np.int64), np.zeros(kt, np.int64), np.zeros(kt, np.float32)\n    k = 0\n    for j, name in enumerate(names):\n        c = bags[name]\n        for i in np.unique(np.linspace(0, len(c) - 1, min(m, len(c))).round().astype(int)) if c else []:\n            si, sl, cy, cx, side, t, pl, ps, _ = c[i]\n            x[k] = crop(series[si]["vol"][[sl - 1, sl, sl + 1]], cy, cx, side, px)\n            mask[k], bag[k], typ[k], plane[k], pos[k] = True, j, t, pl, ps\n            k += 1\n    t = lambda a, dt: torch.from_numpy(np.ascontiguousarray(a[None])).to(dt)\n    return torch.from_numpy(x[None]), t(mask, torch.bool), t(typ, torch.long), t(plane, torch.long), t(pos, torch.float32), t(bag, torch.long)\n\n\ndef correct_sides(series, det, dev, thr=0.9):\n    """Basin\'s mirror detector on each series; flipped back like the training flip table. Returns the number flipped."""\n    n = 0\n    with torch.no_grad():\n        for s in series:\n            if len(s["vol"]) < 3:\n                continue\n            p = torch.sigmoid(det(torch.from_numpy(det_input(s["vol"], s["plane"]))[None].to(dev))).item()\n            if p > thr:\n                n += 1\n                if s["plane"] == "Sagittal":\n                    s["vol"], s["pos"] = s["vol"][::-1].copy(), -np.asarray(s["pos"])[::-1].copy()\n                else:\n                    s["vol"] = s["vol"][..., ::-1].copy()\n    return n\n\n\ndef load_fsmain(members, dev):\n    """members of kind "fsmain": {name: {"kind": "fsmain", "cfg": main train config, "files": [...]}} -> {name: (models, cfg)}."""\n    out = {}\n    for name, e in members.items():\n        if e.get("kind") != "fsmain":\n            continue\n        c = e["cfg"]\n        ms = []\n        for f in e["files"]:\n            m = KneeNet(c["backbone"], n_out=12, pretrained=False, hidden=c["hidden"], drop=c["drop"], use_type=c["use_type"],\n                        use_pos=c["use_pos"], head=c["head"])\n            m.load_state_dict(torch.load(f, map_location="cpu"))\n            ms.append(m.to(dev).to(memory_format=torch.channels_last).eval())\n        out[name] = (ms, c)\n    return out\n\n\n@torch.no_grad()\ndef fsmain_logits(fsm, series, dev):\n    """(Fracture, Contusion) mean logits of each fsmain member on the fat-suppressed series of one study (all series if none)."""\n    out = {}\n    fs = [s for s in series if ki.SERIES_TYPES[s["type"]].endswith("_FS") and len(s["vol"]) >= 3] or series\n    for name, (ms, c) in fsm.items():\n        crop_px = min(PX, int(round(c["crop_mm"] / MM_PER_PX)))\n        slices, idx, mask, typ, pos = ki.windows(fs, c["k_eval"], crop_px, tuple(c["span"]))\n        x = torch.from_numpy(slices).to(dev)[:, None].float().div_(255)\n        x = F.interpolate(x, size=(c["img"], c["img"]), mode="bilinear", antialias=True, align_corners=False)[:, 0]\n        x = x[torch.from_numpy(idx).to(dev).flatten()].view(-1, 3, c["img"], c["img"])\n        cb = ms[0].backbone.pretrained_cfg\n        mean = torch.tensor(cb["mean"], device=dev).view(1, 3, 1, 1)\n        std = torch.tensor(cb["std"], device=dev).view(1, 3, 1, 1)\n        xv = ((x - mean) / std).contiguous(memory_format=torch.channels_last)\n        t = lambda a, dt: torch.from_numpy(np.ascontiguousarray(a)[None]).to(dev, dt)\n        lg = []\n        for m in ms:\n            with torch.autocast(dev.type, dtype=AMP, enabled=dev.type == "cuda"):\n                lg.append(m(xv, t(mask, torch.bool), t(typ, torch.long), t(pos, torch.float32)).float()[0, [FINDINGS_FC["Fracture"], FINDINGS_FC["Contusion"]]])\n        v = torch.stack(lg).mean(0).cpu().numpy()\n        if not np.isfinite(v).all():\n            raise FloatingPointError(f"non-finite fsmain {name} logit")\n        out[name] = v\n    return out\n\n\ndef load_experts(members, dev):\n    """members: {name: {"cfg": bone_train config, "files": [weight paths]}} -> {name: (models, cfg, names, mean, std)}."""\n    experts = {}\n    fs_type = np.array([t.endswith("_FS") for t in ki.SERIES_TYPES])\n    for name, e in members.items():\n        if e.get("kind") == "fsmain":\n            continue\n        c = e["cfg"]\n        names = c["bags"].split(",")\n        ob, ofs = out_masks(c, names)\n        ms = []\n        for f in e["files"]:\n            m = BoneNet(c["backbone"], ob, len(names), len(ki.SERIES_TYPES), ofs, fs_type, c.get("hidden", 256), c.get("bagsum", False), c.get("bb_pool", "avg"))\n            m.load_state_dict(torch.load(f, map_location="cpu"))\n            ms.append(m.to(dev).to(memory_format=torch.channels_last).eval())\n        cfgb = ms[0].backbone.pretrained_cfg\n        mean = torch.tensor(cfgb["mean"], device=dev).view(1, 3, 1, 1)\n        std = torch.tensor(cfgb["std"], device=dev).view(1, 3, 1, 1)\n        experts[name] = (ms, c, names, mean, std)\n    return experts\n\n\nCROP_KEYS = {"bags": None, "planes": "", "series": "all", "trim": None, "crop_scale": 1.0, "loc": None, "pat_margin": MARGIN, "k_eval": None,\n             "px": None, "mono": False}      # crop configuration keys with the defaults study_bags / study_tensors use\n\n\ndef fuse_crops(S, U, groups, gate=20):\n    """oct13_f5_win crop-level fusion (experiments/v2/oct13_deep_ensembling_test/wfuse.py spec \'geo|pm|g20|wmean\'): S, U (models, k,\n    outputs) over the same crops; fold models of one member (group) are averaged first (window probs); relevance = softmax over crops\n    of the mean attention logit over all models; value = mean over members of the window prob; the gate keeps the `gate` crops with\n    the highest relevance per output; pooled prob = relevance-weighted mean of the values -> logit."""\n    r = torch.softmax(S.mean(0), 0)                                            # (k, outputs)\n    P = torch.stack([torch.sigmoid(U[[i for i, g in enumerate(groups) if g == gg]]).mean(0) for gg in sorted(set(groups))]).mean(0)\n    if gate and r.shape[0] > gate:\n        keep = torch.zeros_like(r).scatter_(0, torch.topk(r, gate, 0).indices, 1.0)\n        r = r * keep\n        r = r / r.sum(0, keepdim=True)\n    p = (r * P).sum(0).clamp(1e-6, 1 - 1e-6)\n    return torch.log(p / (1 - p))\n\n\n@torch.no_grad()\ndef expert_logits(experts, series, locs, dev, fuse=None):\n    """fuse (config fc "fuse", oct13_f5_win): also returns out["fused"] = crop-level fusion of every member\'s fold models over the\n    shared crops (members must share the crop configuration)."""\n    out, cache, crops = {}, {}, []\n    for name, (ms, c, names, mean, std) in experts.items():\n        key = tuple(str(c.get(k, d) if c.get(k) is not None else d) for k, d in CROP_KEYS.items()) + (tuple(names),)\n        if key not in cache:                       # members with one crop configuration share the crops (built once)\n            bags = study_bags(series, locs, c)\n            cache[key] = tuple(t.to(dev) for t in study_tensors(series, bags, names, c["k_eval"], c["px"]))\n        x, mask, typ, plane, pos, bag = cache[key]\n        xf = x.flatten(0, 1).float() / 255\n        if c.get("mono", False):                  # data.gpu_prep mono: the centre slice in all three channels (RadImageNet R50)\n            xf = xf[:, 1:2].expand(-1, 3, -1, -1)\n        xf = ((xf - mean) / std).contiguous(memory_format=torch.channels_last)\n        outs = c["outs"].split(",")\n        jj = [outs.index("frac"), outs.index("cont")]\n        lg = []\n        for m in ms:\n            with torch.autocast(dev.type, dtype=AMP, enabled=dev.type == "cuda"):\n                if fuse and not c.get("nofuse", False):     # oct17_probe_cont_fs: a \'nofuse\' member (own crop config) stays out of the fusion\n                    s, u = m(xf, mask, typ, plane, pos, bag, crops=True)\n                    s, u = s.float()[0][:, jj], u.float()[0][:, jj]\n                    n = int(mask[0].sum())\n                    assert bool(mask[0, :n].all()), "crop fusion expects the valid crops first"\n                    crops.append((key, name, s[:n], u[:n]))\n                    lg.append((torch.softmax(s[:n], 0) * u[:n]).sum(0))\n                else:\n                    lg.append(m(xf, mask, typ, plane, pos, bag).float()[0, jj])\n        v = torch.stack(lg).mean(0).cpu().numpy()\n        if not np.isfinite(v).all():\n            raise FloatingPointError(f"non-finite expert {name} logit")\n        out[name] = v\n    if fuse:\n        assert len({k for k, *_ in crops}) == 1, "crop fusion needs one shared crop configuration"   # nofuse members add no crops\n        names = [n for _, n, _, _ in crops]\n        z = fuse_crops(torch.stack([c[2] for c in crops]), torch.stack([c[3] for c in crops]), [sorted(set(names)).index(n) for n in names],\n                       fuse["gate"]).cpu().numpy()\n        if not np.isfinite(z).all():\n            raise FloatingPointError("non-finite fused crop logit")\n        out["fused"] = z\n    return out\n\n\nclass Studies(torch.utils.data.Dataset):\n    def __init__(self, root, studies, fs_of):\n        self.root, self.studies, self.fs_of = root, list(studies), fs_of\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        s = self.studies[i]\n        try:\n            series, _ = ki.study_series(f"{self.root}/{s}", self.fs_of)\n            return s, [{"vol": d["vol"], "plane": d["plane"], "type": int(d["type"]), "pos": np.asarray(d["pos"], np.float32)} for d in series], ""\n        except Exception as e:  # noqa: BLE001\n            return s, [], f"{type(e).__name__}: {e}"\n\n\ndef predict(root, studies, fs_of, members, loc_path, det_path, workers=4, fuse=None):\n    """-> {expert name: DataFrame (study x Fracture, Contusion logits)} (+ "fused" when fuse is set: the crop-level fusion of all\n    members\' fold models); failed studies are missing (STATS[\'failed\'])."""\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    t0 = time.time()\n    loc = Loc()\n    loc.load_state_dict(torch.load(loc_path, map_location="cpu"))\n    loc = loc.to(dev).to(memory_format=torch.channels_last).eval()\n    det = FlipNet(pretrained=False)\n    det.load_state_dict(torch.load(det_path, map_location="cpu"))\n    det = det.to(dev).eval()\n    experts = load_experts(members, dev)\n    fsm = load_fsmain(members, dev)\n    dl = torch.utils.data.DataLoader(Studies(root, studies, fs_of), batch_size=None, num_workers=workers, collate_fn=lambda x: x,\n                                     prefetch_factor=2 if workers else None)\n    rows = {k: {} for k in list(experts) + list(fsm) + (["fused"] if fuse else [])}\n    STATS.update(failed=[], flipped=0)\n    for sid, series, err in dl:\n        try:\n            if err or not series:\n                raise RuntimeError(err or "no readable series")\n            for k, v in fsmain_logits(fsm, series, dev).items():      # main-type members: before the side correction (as trained)\n                rows[k][sid] = v\n            STATS["flipped"] += correct_sides(series, det, dev)\n            locs = [localise(loc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n            for k, v in expert_logits(experts, series, locs, dev, fuse).items():\n                rows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            STATS["failed"].append(sid)\n            print(f"bone_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n    STATS["wall_s"] = time.time() - t0\n    return {k: pd.DataFrame.from_dict(v, orient="index", columns=["Fracture", "Contusion"]) for k, v in rows.items()}\n\n\ndef predict_pkg(root, studies, fs_of, pkg, fc, workers=4):\n    """config.json "fc" block -> {member: DataFrame (study x Fracture, Contusion logits)}; member files, localiser and detector are\n    paths inside the package dir."""\n    members = {n: {"kind": e["kind"], "cfg": e["cfg"], "files": [str(Path(pkg) / f) for f in e["files"]]} for n, e in fc["members"].items()}\n    return predict(root, studies, fs_of, members, str(Path(pkg) / fc["loc"]), str(Path(pkg) / fc["detector"]), workers, fc.get("fuse"))\n\n\ndef apply_recipe(b, member_logits, recipe):\n    """b: DataFrame (study x Fracture, Contusion) = the package\'s F / C logits (mean of its pipelines). Per finding f (recipe\n    "per_finding"): e = mean over its members of (logit - mean) / sd (recipe member_z); o = (1 - w) b + w ((e - mu_e) / sd_e * sd_b + mu_b)\n    (stats from the folds 0-2 OOF). A study without expert logits (stage failure) keeps b."""\n    out = b.copy()\n    for f, r in recipe["per_finding"].items():\n        zs = [(member_logits[m][f] - recipe["member_z"][m][f]["mean"]) / recipe["member_z"][m][f]["sd"] for m in r["members"]]\n        e = (sum(z.reindex(b.index) for z in zs) / len(zs)).astype(np.float64)\n        s = r["stats"]\n        o = (1 - r["w"]) * b[f] + r["w"] * ((e - s["mu_e"]) / s["sd_e"] * s["sd_b"] + s["mu_b"])\n        out[f] = o.where(e.notna(), b[f])\n    return out\n\n\ndef apply_recipe_fused(b, fused, recipe, fuse):\n    """oct13_f5_win: the F / C recipe with its expert term e replaced by the crop-level fusion of all members (config fc "fuse"):\n    e_f = (fused - mu_f) / sd_f * sd_e + mu_e (aligned to the deployed e\'s OOF distribution, folds 0-2), then o = (1 - w) b + w z(e_f)\n    as apply_recipe. A study without a fused logit is NaN here (ensemble.py keeps apply_recipe\'s member-based value)."""\n    out = b.copy()\n    for f, r in recipe["per_finding"].items():\n        al, st = fuse["align"][f], r["stats"]\n        ef = (fused[f].reindex(b.index) - al["mu_f"]) / al["sd_f"] * al["sd_e"] + al["mu_e"]\n        out[f] = (1 - r["w"]) * b[f] + r["w"] * ((ef - st["mu_e"]) / st["sd_e"] * st["sd_b"] + st["mu_b"])\n    return out\n', "check_studies.csv": 'StudyInstanceUID,group\n1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149,gold\n1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269,gold\n1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,gold\n1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580,gold\n1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238,gold\n1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935,gold\n1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840,gold\n1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756,gold\n1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604,gold\n1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581,gold\n1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945,gold\n1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819,gold\n1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973,gold\n1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492,gold\n1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803,gold\n1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891,gold\n1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903,gold\n1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309,gold\n1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192,gold\n1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711,gold\n1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378,gold\n1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404,gold\n1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645,gold\n1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465,gold\n1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924,gold\n1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608,gold\n1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550,gold\n1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911,gold\n1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945,gold\n1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742,gold\n1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683,gold\n1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500,gold\n1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125,gold\n1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187,gold\n1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274,gold\n1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132,gold\n1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450,gold\n1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363,gold\n1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710,gold\n1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395,gold\n1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236,gold\n1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910,gold\n1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039,gold\n1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327,gold\n1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774,gold\n1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002,gold\n1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921,gold\n1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220,gold\n1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995,gold\n1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190,gold\n1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003,gold\n1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828,gold\n1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254,gold\n1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037,gold\n1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435,gold\n1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767,gold\n1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312,gold\n1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662,gold\n1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688,random\n1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868,random\n1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373,random\n1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256,random\n1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184,random\n1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737,random\n1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451,random\n1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569,random\n1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751,random\n1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631,random\n1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548,random\n1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654,random\n1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448,random\n1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365,random\n1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712,random\n1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387,random\n1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018,random\n1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777,random\n1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290,random\n1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734,random\n1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260,edge\n1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585,edge\n1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231,edge\n1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764,edge\n1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542,edge\n1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339,edge\n1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606,edge\n1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513,edge\n1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394,edge\n1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938,edge\n'}
for fn in os.listdir(PKG):
    if fn in ("config.json", "__pycache__", *PATCHED) or os.path.exists(os.path.join(WORK, fn)):
        continue
    if fn.endswith(".py"):       # copies: knee_infer puts Path(__file__).resolve().parent first on sys.path
        shutil.copy(os.path.join(PKG, fn), os.path.join(WORK, fn))
    else:
        os.symlink(os.path.join(PKG, fn), os.path.join(WORK, fn))


def _link(src_dir, files, sub=""):
    for fn in files:
        dst = os.path.join(WORK, sub, fn)
        assert not os.path.exists(dst), dst
        os.symlink(os.path.join(src_dir, fn), dst)


for fn, src in PATCHED.items():
    open(os.path.join(WORK, fn), "w").write(src)
_cfg = json.load(open(PKG + "/config.json"))
_sm = json.load(open(PKG + "/" + _cfg["syn"]["members"]))
for k in ("mcl_member", "mcl_member2", "mcl_sd", "acl_member", "acl_member2"):
    assert k not in _cfg, k
# MCL (Q21): zoom member + SD-2 + b3g area MCL
_mz = json.load(open(ZMEM + "/member.json"))
_link(ZMEM, _mz["entry"]["files"])
_sm[_mz["name"]] = {"cfg": _mz["entry"]["cfg"], "files": _mz["entry"]["files"]}
_sd = json.load(open(SDM + "/member_sd.json"))
_link(SDM, _sd["main_files"] + _sd["nfnet_files"] + [f.replace(".pt", ".json") for f in _sd["nfnet_files"]])
os.makedirs(WORK + "/main_sd", exist_ok=True)
_link(SDM, _sd["main_files"], "main_sd")
json.dump(json.load(open(PKG + "/config.json")) | {"weights": _sd["main_files"]}, open(WORK + "/main_sd/config.json", "w"))
_cfg["mcl_sd"] = {"main_dir": "main_sd", "nfnet": _sd["nfnet_files"], "r50": "rad"}
_cfg["mcl_member"] = _mz["mcl_member"] | _sd["mcl_member_refit"]
assert STACK["name"] == "b3g"
_cfg["mcl_member2"] = STACK | {"name": "b3g" if B3G_SHARED else "b3g_mcl"}
# ACL (Q16): z1_acl + b3g. FULL: b3g_acl (keep notch) + b3g_mcl (keep mcl) = two passes of the same weights (as the probes ran);
# P1: one entry 'b3g' keep 'mcl,notch'
_am = json.load(open(AMEM + "/member.json"))
for _n, _e in _am["members"].items():
    _link(AMEM, _e["files"])
    _c = dict(_e["cfg"])
    if _n == "b3g":
        assert _c["keep_bags"] == "notch", _c["keep_bags"]
        if B3G_SHARED:
            _sm["b3g"] = {"cfg": _c | {"keep_bags": "mcl,notch"}, "files": _e["files"]}
        else:
            _sm["b3g_acl"] = {"cfg": _c, "files": _e["files"]}
            _sm["b3g_mcl"] = {"cfg": _c | {"keep_bags": "mcl"}, "files": _e["files"]}
        continue
    assert _n not in _sm, _n
    _sm[_n] = {"cfg": _c, "files": _e["files"]}
_cfg["acl_member"] = _am["acl_member"]
_cfg["acl_member2"] = _am["acl_member2"] | {"name": "b3g" if B3G_SHARED else "b3g_acl"}
# Synovitis: the ConvNeXt crop slots (the effb0 crop members stay for Effusion's img_eff = eff_pipelines)
_cm = json.load(open(CMEM + "/syn_members.json"))
_cnx = sorted(k for k in _cm if k.startswith("cs_"))
assert len(_cnx) == 6, _cnx
for _n in _cnx:
    _link(CMEM, _cm[_n]["files"])
    assert _n not in _sm, _n
    _sm[_n] = _cm[_n]
_syn = _cfg["syn"]
_syn["eff_pipelines"] = json.loads(json.dumps(_syn["pipelines"]))
_syn["pipelines"]["main"] = [n.replace("s_fs_", "cs_fs_").replace("s_fc125_", "cs_fc125_") for n in _syn["pipelines"]["main"]]
for _d in _syn["fuse"]["designs"].values():
    _d["main"] = [n.replace("s_fs_", "cs_fs_").replace("s_fc125_", "cs_fc125_") for n in _d["main"]]
assert set(_syn["pipelines"]["main"]) == set(_cnx), _syn["pipelines"]["main"]
if P2_EFF:   # P2: the six effb0 crop members served only Effusion's img_eff -> taken from the ConvNeXt crops; the effb0 members leave the stack
    _syn["eff_pipelines"]["main"] = list(_syn["pipelines"]["main"])
    _drop = [n for n in _sm if n.startswith(("s_fs_f", "s_fc125_f"))]
    assert len(_drop) == 6, _drop
    for _n in _drop:
        del _sm[_n]
    assert all(n in _sm for ns in _syn["eff_pipelines"].values() for n in ns) and all(n in _sm for ns in _syn["pipelines"].values() for n in ns)
    _cfg["eff"]["stats"]["Effusion"].update(P2_EFF)
json.dump(_sm, open(WORK + "/syn_members_stack.json", "w"))
_syn["members"] = "syn_members_stack.json"
# LOA (Q15): the upgraded OA groups for LOA only
_lm = json.load(open(LMEM + "/member.json"))
_le = _lm["members"]
for _e in _le.values():
    _link(LMEM, _e["files"])
assert not set(_le) & set(_cfg["oa"]["members"]) and "loa_extra" not in _cfg["oa"]
_cfg["oa"]["members"].update(_le)
_cfg["oa"]["loa_extra"], _cfg["oa"]["loa_set"], _cfg["oa"]["loa_extra_align"] = list(_le), _lm["loa_set"], _lm["align"]
# Contusion FS (LB-won): the bfs_fs bone member (nofuse) + the cont_fs splice constants (oct17_probe_cont_fs)
_cf = json.load(open(CFM + "/member.json"))
_link(CFM, _cf["entry"]["files"])
assert _cf["name"] not in _cfg["fc"]["members"] and "cont_fs" not in _cfg and _cf["entry"]["cfg"].get("nofuse"), "cont_fs config"
_cfg["fc"]["members"][_cf["name"]] = {"kind": "bone", "cfg": _cf["entry"]["cfg"], "files": _cf["entry"]["files"]}
_cfg["cont_fs"] = _cf["cont_fs"]
if SWITCHES["frac_mfs"]:          # oct19_probe_frac_mfs
    _mf = json.load(open(MFM + "/member.json"))
    _link(MFM, _mf["entry"]["files"])
    assert _mf["name"] not in _cfg["fc"]["members"] and "frac_mfs" not in _cfg and _mf["entry"]["kind"] == "fsmain", "frac_mfs config"
    _cfg["fc"]["members"][_mf["name"]] = {"kind": "fsmain", "cfg": _mf["entry"]["cfg"], "files": _mf["entry"]["files"]}
    _cfg["frac_mfs"] = _mf["frac_mfs"]
if SWITCHES["pf_q20"]:            # oct18_probe_pf_allup: PF from the fusion over ONLY its loa_set (members already in oa.members for Q15)
    assert set(PF_Q20["set"]) <= set(_cfg["oa"]["members"]) and PF_Q20["col"] == "PF OA", PF_Q20
    _cfg["oa"]["extra_sets"] = {"pf": PF_Q20["set"]}
    _cfg["oa"]["extra_align"] = {"pf": PF_Q20["align"] | {"col": PF_Q20["col"]}}
if SWITCHES["bakers_allpw"]:      # oct19_probe_bakers_allpw (Q25)
    _bk = json.load(open(BKM + "/member.json"))["bakers_allpw"]
    _link(BKM, _bk["main"] + [c for cs in _bk["basin"].values() for c in cs] + [c[:-3] + ".json" for cs in _bk["basin"].values() for c in cs])
    assert "bakers_allpw" not in _cfg and _bk["cols"] == ["Baker's"] and len(_bk["main"]) == 5 and all(len(v) == 5 for v in _bk["basin"].values())
    _cfg["bakers_allpw"] = {"main": _bk["main"], "basin": _bk["basin"], "cols": _bk["cols"]}
if SWITCHES["lm_allpw"]:          # C184 G1: LM's 3x3 base from Q25's pw generalists (pw main pass in its own weights dir + the pw Basin models)
    _lb = json.load(open(BKM + "/member.json"))["bakers_allpw"]
    _bf = [c for cs in _lb["basin"].values() for c in cs]
    _link(BKM, [f for f in _bf + [c[:-3] + ".json" for c in _bf] if not os.path.exists(os.path.join(WORK, f))])   # shared with bakers_allpw
    os.makedirs(WORK + "/main_pw", exist_ok=True)
    _link(BKM, _lb["main"], "main_pw")
    json.dump(json.load(open(PKG + "/config.json")) | {"weights": _lb["main"]}, open(WORK + "/main_pw/config.json", "w"))
    assert "lm_allpw" not in _cfg and len(_lb["main"]) == 5 and set(_lb["basin"]) == set(_cfg["basin"]) and all(len(v) == 5 for v in _lb["basin"].values())
    _cfg["lm_allpw"] = {"main_dir": "main_pw", "basin": _lb["basin"], "col": "Lateral Meniscus"}
json.dump(_cfg, open(WORK + "/config.json", "w"))
PKG = WORK
print("stack: syn members", len(_sm), "| mcl_sd", _cfg["mcl_sd"], "| mcl_member", _cfg["mcl_member"]["name"], "| mcl_member2", _cfg["mcl_member2"]["name"],
      "| acl_member", _cfg["acl_member"]["name"], "| acl_member2", _cfg["acl_member2"]["name"], "| syn main", _syn["pipelines"]["main"],
      "| loa_set", _cfg["oa"]["loa_set"], "| fc members", list(_cfg["fc"]["members"]), "| cont_fs", _cfg["cont_fs"]["name"], "| SWITCHES", SWITCHES,
      "| oa extra_sets", _cfg["oa"].get("extra_sets"), "| bakers_allpw", bool(_cfg.get("bakers_allpw")), "| lm_allpw", bool(_cfg.get("lm_allpw")), flush=True)
sys.path.insert(0, PKG)
import cv2, pydicom, timm
import ensemble
ki = ensemble.ki
for _m in (ensemble, ensemble.syn_stage, ensemble.oa_stage, ensemble.fc_stage, ki):     # the stack code is what runs (fail fast otherwise)
    assert os.path.dirname(os.path.abspath(_m.__file__)) == WORK, _m.__file__
_es = open(ensemble.__file__).read()
assert all(k in _es for k in ("mcl_member2", "mcl_sd", "acl_member2", "loa_extra_align", "eff_pipelines", "cont_fs", "frac_mfs", "extra_sets", "def basin_args", "lm_allpw")) \
    and "keep_bags" in open(ensemble.syn_stage.__file__).read() and "nofuse" in open(ensemble.fc_stage.__file__).read() \
    and "extra_sets" in open(ensemble.oa_stage.__file__).read()
print("stack modules loaded from", WORK, flush=True)
NCPU = len(os.sched_getaffinity(0))
print(f"python {platform.python_version()} torch {torch.__version__} timm {timm.__version__} pydicom {pydicom.__version__} cv2 {cv2.__version__} "
      f"numpy {np.__version__} pandas {pd.__version__} | cpus {NCPU} gpus {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]} "
      f"| package {PKG} | weights {json.load(open(PKG + '/config.json'))['weights']} basin {json.load(open(PKG + '/config.json'))['basin']}", flush=True)

DEC = {'MCL': {'context': 'Contusion', 'b': 0.2, 'c0': -2.0172928736727602, 'c': 0.31754119357063526}, 'Lateral Meniscus': {'context': 'ACL', 'b': 0.1, 'c0': -1.371208412532161, 'c': 0.322331809283129}}                                          # post-hoc de-coupling z' = z - b (c0 + c z_ctx): MCL Q6 RE-FITTED on the new Contusion (refit.json), LM Q22


def decouple(df):
    out = df.copy()
    for f, d in DEC.items():
        p, q = np.clip(df[f].to_numpy(np.float64), 1e-7, 1 - 1e-7), np.clip(df[d["context"]].to_numpy(np.float64), 1e-7, 1 - 1e-7)
        z, zc = np.log(p / (1 - p)), np.log(q / (1 - q))
        out[f] = 1 / (1 + np.exp(-(z - d["b"] * (d["c0"] + d["c"] * zc))))
    return out


test = pd.read_csv(COMP + "/test.csv")
t = time.time()
# ===== FINAL fold-avg package + FAST two-GPU runner, STRICT (Master 10-20, submissions/oct20_final_v2/make_final.py; OAI: no).
# USER: "we absolutely do not need any fallbacks. we need robust inference pipeline that works or fails loudly".
import gc
open("/kaggle/working/fast.py", "w").write('"""Two-GPU runner, port to the merged stacked fold package submissions/oct20_stack_all (from submissions/oct19_f5_win_fast, which was\nbit-identical to the 0.961 package on Kaggle). The workers take their stage arguments from the package\'s ensemble.py hooks\n(basin_args / oa_cfg / fc_cfg / syn_members: the same functions ensemble.predict calls), so every switch (frac_mfs / pf_q20 / bakers_allpw)\ngives identical stage calls. Dev runs against oct19_stack_fold use a hook shim (env FAST_HOOKS). Serial test mode (env FAST_SERIAL=1):\nthe two workers run one after another on GPU 0 (local parity on a 1-GPU machine; same outputs as two GPUs of one type).\n\nOriginal header (0.961 runner): Predictions = the package\'s own.\n\nThe package runs its stages one after another, and four of them use one T4 only (Basin, OA experts, bone experts, synovitis crops: ~90 % of\nits 262 min). Here two worker processes (CUDA_VISIBLE_DEVICES 0 / 1) each run those four stages on half of the studies, at the same time.\nThe halves follow Basin\'s study order (the series csv order) and split at an even index, so Basin\'s 2-study batches are the package\'s.\nknee_infer (main) and the window stage already use both GPUs: they stay in the parent. The parent then calls the package\'s unchanged\nensemble.predict with the four stage functions returning the workers\' merged outputs, so the combination code is the package\'s own.\nOne difference by construction: Basin\'s fallback for a study without any readable series (cohort mean logit) is the mean over that\nworker\'s half instead of over all studies (never triggered on the 275 check studies of 10-13).\n\nProfiling (prof=True): every stage call\'s start / end (parent and workers), and the time inside the crop stages\' building blocks\n(knee_infer.study_series in the DataLoader workers, localise, study_bags, study_tensors, correct_sides, expert / fsmain logits);\nread with read_prof(). Timing wrappers only: no change to any call\'s arguments or results.\n\nRAM (10-19, after the ON bench\'s host OOM): every worker stage starts and ends with release() (gc, CUDA cache, pinned-host cache, glibc\narenas; RSS logged in the worker timings), the crop loop releases the pinned-host cache every RELEASE_EVERY studies, pad_loc 32 is the\ndefault, and every predict() call spawns fresh worker processes that exit at its end (all their memory returns to the OS).\n\n  worker: python fast.py worker PKG ROOT JOB.pkl OUT.pkl"""\nimport contextlib\nimport json\nimport os\nimport pickle\nimport resource\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nSTAGE = ["parent"]          # current stage label; DataLoader workers inherit it at fork\nPROF_DIR = os.environ.get("KNEE_PROF_DIR", "/kaggle/working/prof")\nSTAGE_LOG = []              # (process, stage, t_start, t_end)\nRELEASE_EVERY = 16          # crop loop: return the pinned-host cache / glibc arenas every N studies (10-19 RAM fix; no effect on outputs)\nPARENT_RELEASE_EVERY = 16   # parent: the same for knee_infer\'s main passes (10-19 Kaggle parent_mem: 25 GB of cached pinned study buffers on 325 studies)\n\n\ndef rss_gb():\n    return int(next(l for l in open("/proc/self/status") if l.startswith("VmRSS")).split()[1]) / 2**20\n\n\n@contextlib.contextmanager\ndef parent_release(ki, every=PARENT_RELEASE_EVERY):\n    """Parent-side pinned-cache release (10-19; Kaggle parent_mem: knee_infer\'s cached pinned study buffers reached 25 GB in one main pass of\n    325 heavy studies, all returned by the package\'s own _host_emptyCache AFTER the pass). Every `every` knee_infer.load_study calls (they run\n    in knee_infer\'s loader threads) the cache\'s FREE pinned blocks are returned; blocks in use or with pending copy events stay (the host\n    allocator is mutex-protected and frees only unused, event-complete blocks). Memory only: outputs unchanged."""\n    import torch\n    orig, n, lock = ki.load_study, [0], threading.Lock()\n\n    def wrapped(*a, **k):\n        r = orig(*a, **k)\n        with lock:\n            n[0] += 1\n            go = every and n[0] % every == 0\n        if go and torch.cuda.is_available():\n            torch._C._host_emptyCache()\n        return r\n    ki.load_study = wrapped\n    try:\n        yield n\n    finally:\n        ki.load_study = orig\n\n\ndef release(full=True):\n    """Return freed memory (10-19 RAM fix after the ON bench\'s host OOM; no effect on any output): Python garbage (full), the CUDA caching\n    allocator (full), the pinned-host cache (grows with every new pinned buffer size: knee_infer pins each study\'s DICOM bytes) and glibc\'s\n    malloc arenas."""\n    import ctypes\n    import gc\n    import torch\n    if full:\n        gc.collect()\n    if torch.cuda.is_available():\n        if full:\n            torch.cuda.empty_cache()\n        torch._C._host_emptyCache()\n    ctypes.CDLL("libc.so.6").malloc_trim(0)\nCROP_FUNCS = ["localise", "study_bags", "study_tensors", "correct_sides", "expert_logits", "fsmain_logits"]\n\n\ndef _mods(pkg):\n    sys.path.insert(0, pkg)\n    import basin_predict as bp\n    import fc_stage\n    import knee_infer as ki\n    import oa_stage\n    import syn_stage\n    import win_stage\n    return bp, ki, oa_stage, fc_stage, syn_stage, win_stage\n\n\ndef _hooks(pkg):\n    """The package\'s stage-argument hooks (ensemble.py). FAST_HOOKS=path/to/shim.py only for dev against a package without them."""\n    sys.path.insert(0, pkg)\n    import importlib.util\n    shim = os.environ.get("FAST_HOOKS")\n    if shim:\n        spec = importlib.util.spec_from_file_location("fast_hooks", shim)\n        mod = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(mod)\n        return mod\n    import ensemble\n    assert all(hasattr(ensemble, h) for h in ("basin_args", "oa_cfg", "fc_cfg", "syn_members")), "ensemble.py lacks the stage hooks"\n    return ensemble\n\n\ndef _append(name, sec):\n    os.makedirs(PROF_DIR, exist_ok=True)\n    with open(f"{PROF_DIR}/prof_{os.getpid()}.tsv", "a") as fh:\n        fh.write(f"{STAGE[0]}\\t{name}\\t{sec:.6f}\\n")\n\n\ndef _timed(fn, name):\n    def w(*a, **k):\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            _append(name, time.time() - t)\n    w.__wrapped__ = fn\n    return w\n\n\ndef _stage(fn, label, proc):\n    def w(*a, **k):\n        prev, STAGE[0] = STAGE[0], label\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            STAGE_LOG.append((proc, label, t, time.time()))\n            STAGE[0] = prev\n    w.__wrapped__ = fn\n    return w\n\n\ndef _timed_sync(fn, name):\n    """Timing wrapper that synchronises the GPU before and after (profiling runs only: adds two syncs per call)."""\n    import torch\n\n    def w(*a, **k):\n        torch.cuda.synchronize()\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            torch.cuda.synchronize()\n            _append(name, time.time() - t)\n    w.__wrapped__ = fn\n    return w\n\n\ndef install_prof(pkg, proc, sync=False):\n    """Timing wrappers on the package modules (idempotent). sync: also Basin\'s gpu_views / export_windows / model forwards with GPU\n    syncs (their GPU time; changes scheduling, not results)."""\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    if sync and getattr(bp.export_windows, "__wrapped__", None) is None:\n        bp.export_windows = _timed_sync(bp.export_windows, "basin.export_windows")\n        bp.gpu_views = _timed_sync(bp.gpu_views, "basin.gpu_views")\n        orig_load = bp.load_model\n\n        def load_model(ckpt):\n            m, cfg = orig_load(ckpt)\n            m.forward = _timed_sync(m.forward, "basin.model_forward")\n            return m, cfg\n        bp.load_model = load_model\n    if getattr(ki.study_series, "__wrapped__", None) is None:\n        ki.study_series = _timed(ki.study_series, "study_series")\n        for mod in (oa_stage, fc_stage, syn_stage):\n            for f in CROP_FUNCS:\n                if hasattr(mod, f):\n                    setattr(mod, f, _timed(getattr(mod, f), f))\n        for mod, f, label in ((bp, "predict", "basin"), (ki, "predict", "main"), (win_stage, "main_windows", "win"),\n                              (oa_stage, "predict", "oa"), (fc_stage, "predict_pkg", "fc"), (syn_stage, "predict", "syn")):\n            setattr(mod, f, _stage(getattr(mod, f), label, proc))\n\n\ndef install_pad_loc(pkg, bucket):\n    """The crop stages\' localiser (a per-slice MobileNetV3 + MLP head, BatchNorm in eval mode) on series batches padded to a multiple of\n    `bucket` slices (zero rows, outputs dropped): a few fixed batch shapes instead of one per series length (Kaggle\'s cuDNN builds kernels\n    for every new shape; bench v1: the OA stage\'s localise took 78 s for 101 calls, the same calls 2.6 s in later stages). Real rows\'\n    outputs can differ only by cuDNN\'s per-shape fp32 algorithm choice. NOT exact: a separate bench variant."""\n    import cv2\n    import numpy as np\n    import torch\n    bp, ki, oa_stage, fc_stage, syn_stage, _ = _mods(pkg)\n    for mod in (oa_stage, fc_stage, syn_stage):\n        @torch.no_grad()\n        def localise(loc, vol, plane, pos, dev, mod=mod):\n            n = len(vol)\n            small = np.stack([cv2.resize(a, (mod.LOC_PX, mod.LOC_PX), interpolation=cv2.INTER_AREA) for a in vol])\n            w = np.stack([small[np.clip(np.arange(n) + d, 0, n - 1)] for d in (-1, 0, 1)], 1)\n            aux = mod.aux_features(plane, pos, n)\n            m = -(-n // bucket) * bucket\n            if m > n:\n                w = np.concatenate([w, np.zeros((m - n,) + w.shape[1:], w.dtype)])\n                aux = np.concatenate([aux, np.zeros((m - n,) + aux.shape[1:], aux.dtype)])\n            p, b = loc(torch.from_numpy(w).to(dev), torch.from_numpy(aux).to(dev))\n            return torch.sigmoid(p[:n]).float().cpu().numpy(), b[:n].float().cpu().numpy()\n        mod.localise = localise\n\n\ndef read_prof():\n    """-> DataFrame stage x function -> total seconds, calls (all processes\' files)."""\n    rows = []\n    for p in Path(PROF_DIR).glob("prof_*.tsv"):\n        d = pd.read_csv(p, sep="\\t", names=["stage", "func", "sec"])\n        d["pid"] = p.stem[5:]\n        rows.append(d)\n    if not rows:\n        return pd.DataFrame()\n    d = pd.concat(rows)\n    return d.groupby(["stage", "func"]).sec.agg(["sum", "count"]).round(2)\n\n\ndef worker(pkg, root, job, out):\n    """The package\'s four single-GPU stage calls (arguments as ensemble.predict builds them) on this worker\'s studies."""\n    t_enter = time.time()\n    J = pickle.load(open(job, "rb"))\n    if J.get("pad_loc"):\n        install_pad_loc(pkg, J["pad_loc"])\n    if J["prof"]:\n        install_prof(pkg, J["name"], sync=J.get("prof_sync", False))\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    import torch\n    T = {"t_enter": t_enter, "import_s": time.time() - t_enter}       # spawn -> enter = t_enter - FAST["t0"] (per-call overhead)\n    FINDINGS = ki.FINDINGS\n    cfg = json.load(open(Path(pkg) / "config.json"))\n    studies, workers = J["studies"], J["workers"]\n    series = J["series"]\n    series = series[series.StudyInstanceUID.isin(studies)].assign(Fat_Suppression=lambda d: d.Fat_Suppression.fillna(0).astype(int))\n    fs_of = dict(zip(series.SeriesInstanceUID, series.Fat_Suppression))\n    R = {}\n\n    H = _hooks(pkg)\n\n    def basin():\n        ckpts, det, win_kw = H.basin_args(cfg, pkg)\n        z = bp.predict(series, root, ckpts, workers=workers, detector=det, win=win_kw) if len(series) else {}\n        return z, dict(bp.WIN), dict(bp.STATS)\n\n    def oa():\n        return oa_stage.predict(root, studies, fs_of, pkg, H.oa_cfg(cfg), workers=workers, threads=None), dict(oa_stage.STATS)\n\n    def fc():\n        return fc_stage.predict_pkg(root, studies, fs_of, pkg, H.fc_cfg(cfg), workers=workers), dict(fc_stage.STATS)\n\n    def syn():\n        s = cfg["syn"]\n        members = H.syn_members(cfg, pkg)\n        zm = pd.DataFrame(syn_stage.predict(root, studies, fs_of, members, str(Path(pkg) / s["loc"]), str(Path(pkg) / s["detector"]),\n                                            workers=workers, fuse=s.get("fuse")))\n        return zm, dict(syn_stage.STATS)\n\n    run = {"basin": basin, "oa": oa, "fc": fc, "syn": syn}\n    have = {"basin": True, "oa": bool(cfg.get("oa")) and H.oa_cfg(cfg)["mode"] != "main", "fc": bool(cfg.get("fc")), "syn": bool(cfg.get("syn"))}\n    order = J.get("order", ["basin", "oa", "fc", "syn"])\n    if J.get("merge_crops"):              # one study loop for the three crop stages (crops), at the position of the first crop stage\n        assert all(have[s] for s in ("oa", "fc", "syn")), "merge_crops needs the oa, fc and syn stages"\n        first = min(order.index(s) for s in ("oa", "fc", "syn"))\n        order = [s for s in order if s not in ("oa", "fc", "syn")]\n        order.insert(first, "crops")\n        have["crops"] = True\n        run["crops"] = lambda: crop_all(pkg, root, studies, fs_of, cfg, workers)\n    for st in order:\n        if have[st]:\n            release()                         # the previous stage\'s models / buffers are unreferenced locals by now: give their memory back\n            T[f"{st}_rss_in_gb"] = rss_gb()\n            t = time.time()\n            r = run[st]()\n            if st == "crops":\n                R.update(r)\n                T["crops_load_s"] = r["oa"][1]["load_s"]\n            else:\n                R[st] = r\n            T[st] = time.time() - t\n            T[f"{st}_rss_out_gb"] = rss_gb()\n            release()\n            T[f"{st}_rss_released_gb"] = rss_gb()\n    T["rss_peak_gb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20\n    T["rss_peak_loader_gb"] = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 2**20\n    T["gpu_peak_gb"] = torch.cuda.max_memory_reserved() / 2**30\n    pickle.dump({"R": R, "T": T, "stage_log": STAGE_LOG}, open(out, "wb"))\n\n\ndef crop_all(pkg, root, studies, fs_of, cfg, workers):\n    """oa_stage.predict + fc_stage.predict_pkg + syn_stage.predict in ONE study loop: each study\'s DICOM is decoded once (one\n    DataLoader), the shared mirror detector (one weight file, identical correct_sides in the three modules) runs once, then each stage\'s\n    own localiser and experts run as in its predict(), with that stage\'s failure handling. The bone stage\'s main-type members still see\n    the uncorrected series. -> {"oa": (out, STATS), "fc": (out, STATS), "syn": (DataFrame, STATS)} in the stages\' return formats."""\n    import collections\n\n    import numpy as np\n    import torch\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    from basin_flipdet import FlipNet\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    t0 = time.time()\n    H = _hooks(pkg)\n    ocfg, fcfg, scfg = H.oa_cfg(cfg), H.fc_cfg(cfg), cfg["syn"]\n    dets = {str(Path(pkg) / ocfg["detector"]), str(Path(pkg) / fcfg["detector"]), str(Path(pkg) / scfg["detector"])}\n    blobs = {open(p, "rb").read() for p in dets}\n    assert len(blobs) == 1, "merge_crops needs one detector for the three crop stages"\n    assert oa_stage.correct_sides.__code__.co_code == fc_stage.correct_sides.__code__.co_code == syn_stage.correct_sides.__code__.co_code \\\n        or all(hasattr(f, "__wrapped__") for f in (oa_stage.correct_sides, fc_stage.correct_sides, syn_stage.correct_sides))\n    det = FlipNet(pretrained=False)\n    det.load_state_dict(torch.load(sorted(dets)[0], map_location="cpu"))\n    det = det.to(dev).eval()\n    # OA (as oa_stage.predict)\n    assert all(m["kind"] == "expert" for m in ocfg["members"].values()), "merge_crops: OA main-type members not supported"\n    cor_v2 = ocfg.get("loc_version", "v1") == "v2cor"\n    structs = oa_stage.STRUCTS_V2 if cor_v2 else oa_stage.STRUCTS\n    oloc = oa_stage.Loc(structs=structs)\n    oloc.load_state_dict(torch.load(f"{pkg}/{ocfg[\'loc\']}", map_location="cpu"))\n    oloc = oloc.to(dev).to(memory_format=torch.channels_last).eval()\n    oexp = oa_stage.load_experts(pkg, ocfg, dev)\n    xsets = {k: frozenset(v) for k, v in (ocfg.get("extra_sets") or {}).items()}     # oct20: oa_stage.predict\'s extra fusions\n    orows = {k: {} for k in list(oexp) + (["fused"] if ocfg.get("fuse") else []) + (["fused_loa_extra"] if ocfg.get("loa_extra") else [])\n             + [f"fused_{k}" for k in xsets]}\n    okw = {"extra_sets": xsets} if xsets else {}\n    # bone (as fc_stage.predict_pkg / predict)\n    fmem = {n: {"kind": e["kind"], "cfg": e["cfg"], "files": [str(Path(pkg) / f) for f in e["files"]]} for n, e in fcfg["members"].items()}\n    floc = fc_stage.Loc()\n    floc.load_state_dict(torch.load(str(Path(pkg) / fcfg["loc"]), map_location="cpu"))\n    floc = floc.to(dev).to(memory_format=torch.channels_last).eval()\n    fexp, fsm = fc_stage.load_experts(fmem, dev), fc_stage.load_fsmain(fmem, dev)\n    frows = {k: {} for k in list(fexp) + list(fsm) + (["fused"] if fcfg.get("fuse") else [])}\n    # synovitis (as syn_stage.predict)\n    smem = H.syn_members(cfg, pkg)\n    sloc = syn_stage.Loc()\n    sloc.load_state_dict(torch.load(str(Path(pkg) / scfg["loc"]), map_location="cpu"))\n    sloc = sloc.to(dev).to(memory_format=torch.channels_last).eval()\n    sexp = syn_stage.load_experts(smem, dev)\n    srows = collections.defaultdict(dict)\n    S = {k: {"failed": [], "flipped": 0} for k in ("oa", "fc", "syn")}\n    load_s = time.time() - t0                  # the three stages\' models + localisers + detector loaded (per-call overhead)\n    dl = torch.utils.data.DataLoader(oa_stage.Studies(root, studies, fs_of), batch_size=None, num_workers=workers, collate_fn=lambda x: x,\n                                     prefetch_factor=2 if workers else None)\n    for i_study, (sid, series, err) in enumerate(dl):\n        if RELEASE_EVERY and i_study and i_study % RELEASE_EVERY == 0:\n            release(full=False)\n        if err or not series:\n            for k in S:\n                S[k]["failed"].append(sid)\n            print(f"crops: study {sid} failed ({err or \'no readable series\'})", flush=True)\n            continue\n        try:\n            for k, v in fc_stage.fsmain_logits(fsm, series, dev).items():   # main-type bone members: before the side correction (as trained)\n                frows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["fc"]["failed"].append(sid)\n            print(f"bone_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n            fc_failed = True\n        else:\n            fc_failed = False\n        try:\n            n_flip = oa_stage.correct_sides(series, det, dev)\n        except Exception as e:  # noqa: BLE001\n            for k in S:\n                if not (k == "fc" and fc_failed):\n                    S[k]["failed"].append(sid)\n            print(f"crops: study {sid} failed at the side correction ({type(e).__name__}: {e})", flush=True)\n            continue\n        for k in S:\n            S[k]["flipped"] += n_flip\n        try:\n            bags = oa_stage.study_bags(series, oloc, dev, structs, cor_v2)\n            for k, v in oa_stage.expert_logits(oexp, series, bags, dev, ocfg.get("fuse"), set(ocfg.get("loa_extra", [])),\n                                               set(ocfg.get("loa_set", [])), **okw).items():   # = the stack\'s oa_stage.predict call\n                orows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["oa"]["failed"].append(sid)\n            print(f"oa_stage: study {sid} failed ({type(e).__name__}: {e}); OA keeps the pipeline blend", flush=True)\n        if not fc_failed:\n            try:\n                locs = [fc_stage.localise(floc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n                for k, v in fc_stage.expert_logits(fexp, series, locs, dev, fcfg.get("fuse")).items():\n                    frows[k][sid] = v\n            except Exception as e:  # noqa: BLE001\n                S["fc"]["failed"].append(sid)\n                print(f"bone_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n        try:\n            locs = [syn_stage.localise(sloc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n            for k, v in syn_stage.expert_logits(sexp, series, locs, dev, scfg.get("fuse")).items():\n                srows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["syn"]["failed"].append(sid)\n            print(f"syn_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n    wall = time.time() - t0\n    for k in S:\n        S[k]["wall_s"], S[k]["load_s"] = wall, load_s\n    oout = {k: pd.DataFrame.from_dict(v, orient="index", columns=oa_stage.OUTS).astype(np.float64) for k, v in orows.items()}\n    fout = {k: pd.DataFrame.from_dict(v, orient="index", columns=["Fracture", "Contusion"]) for k, v in frows.items()}\n    sout = pd.DataFrame({k: pd.Series(v, dtype=np.float64) for k, v in srows.items()})\n    return {"oa": (oout, S["oa"]), "fc": (fout, S["fc"]), "syn": (sout, S["syn"])}\n\n\ndef split(studies, series):\n    """Two study lists: Basin\'s order (series csv order of the studies), cut at an even index; studies without series rows go to the\n    first list (Basin never sees them; the crop stages log them as failed, as in the package)."""\n    order = list(dict.fromkeys(series.StudyInstanceUID[series.StudyInstanceUID.isin(studies)]))\n    k = (len(order) + 1) // 2\n    k += k % 2\n    missing = [s for s in studies if s not in set(order)]\n    return [order[:k] + missing, order[k:]]\n\n\ndef _merge(parts):\n    """Worker outputs -> one stage output per stage, in the package\'s return formats."""\n    out = {}\n    zs = [p["R"]["basin"] for p in parts]\n    keys = list(dict.fromkeys(k for z, _, _ in zs for k in z))\n    out["basin"] = ({k: pd.concat([z[k] for z, _, _ in zs if k in z]) for k in keys},\n                    {s: v for _, w, _ in zs for s, v in w.items()},\n                    {"workers": [st for _, _, st in zs]})\n    for st in ("oa", "fc"):\n        if all(st in p["R"] for p in parts):\n            ms = [p["R"][st][0] for p in parts]\n            out[st] = ({k: pd.concat([m[k] for m in ms if k in m]) for k in dict.fromkeys(k for m in ms for k in m)},\n                       {"failed": sum((p["R"][st][1].get("failed", []) for p in parts), []),\n                        "flipped": sum(p["R"][st][1].get("flipped", 0) for p in parts),\n                        "wall_s": max(p["R"][st][1].get("wall_s", 0) for p in parts)})\n    if all("syn" in p["R"] for p in parts):\n        out["syn"] = (pd.concat([p["R"]["syn"][0] for p in parts]),\n                      {"failed": sum((p["R"]["syn"][1].get("failed", []) for p in parts), []),\n                       "flipped": sum(p["R"]["syn"][1].get("flipped", 0) or 0 for p in parts)})\n    return out\n\n\ndef predict(root, studies, series, pkg, threads=None, workers=2, prof=False, work="/kaggle/working/fast_jobs", log=print,\n            orders=(("basin", "oa", "fc", "syn"), ("basin", "oa", "fc", "syn")), merge_crops=False, pad_loc=32, prof_sync=False):\n    """= ensemble.predict(root, studies, series, pkg, threads) with the four single-GPU stages on two GPUs. Returns its 4-tuple; FAST\n    holds the worker / stage timings. orders: the stage order per worker (independent stages: outputs do not depend on it)."""\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    import ensemble\n    import torch\n    if prof:\n        install_prof(pkg, "parent")\n    serial = os.environ.get("FAST_SERIAL") == "1"           # local parity: both workers on GPU 0, one after another\n    ngpu = torch.cuda.device_count()\n    assert serial or ngpu >= 2, f"fast.predict needs 2 GPUs, found {ngpu}"\n    os.makedirs(work, exist_ok=True)\n    t0 = time.time()\n    procs = []\n    for g, part in enumerate(split(studies, series)):\n        job, out = f"{work}/job{g}.pkl", f"{work}/out{g}.pkl"\n        pickle.dump({"studies": part, "series": series, "workers": workers, "prof": prof, "name": f"gpu{g}", "order": list(orders[g]), "merge_crops": merge_crops, "pad_loc": pad_loc, "prof_sync": prof_sync},\n                    open(job, "wb"))\n        if os.path.exists(out):\n            os.remove(out)\n        fh = open(f"{work}/worker{g}.log", "w")\n        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0" if serial else str(g)}\n        procs.append((subprocess.Popen([sys.executable, __file__, "worker", pkg, root, job, out], env=env, stdout=fh, stderr=subprocess.STDOUT), out, fh, len(part)))\n        if serial:\n            procs[-1][0].wait()\n    for p, _, fh, _ in procs:\n        p.wait()\n        fh.close()\n    t_par = time.time() - t0\n    for g, (p, out, _, n) in enumerate(procs):\n        if p.returncode != 0 or not os.path.exists(out):\n            raise RuntimeError(f"fast worker gpu{g} failed (code {p.returncode}); log tail:\\n" + open(f"{work}/worker{g}.log").read()[-4000:])\n    parts = [pickle.load(open(out, "rb")) for _, out, _, _ in procs]\n    M = _merge(parts)\n    log(f"fast: workers done in {t_par:.0f}s ({\', \'.join(f\'gpu{g} {n} studies \' + str({k: round(v) for k, v in pp[\'T\'].items()}) for g, ((_, _, _, n), pp) in enumerate(zip(procs, parts)))})")\n    STAGE_LOG.extend(e for pp in parts for e in pp["stage_log"])\n\n    orig = {"bp": bp.predict, "oa": oa_stage.predict, "fc": fc_stage.predict_pkg, "syn": syn_stage.predict}\n\n    def bp_fake(*a, **k):\n        z, w, st = M["basin"]\n        bp.WIN.clear()\n        bp.WIN.update(w)\n        bp.STATS.clear()\n        bp.STATS.update(st)\n        return z\n\n    def oa_fake(*a, **k):\n        m, st = M["oa"]\n        oa_stage.STATS.clear()\n        oa_stage.STATS.update(st)\n        return dict(m)\n\n    def fc_fake(*a, **k):\n        m, st = M["fc"]\n        fc_stage.STATS.clear()\n        fc_stage.STATS.update(st)\n        return dict(m)\n\n    def syn_fake(*a, **k):\n        zm, st = M["syn"]\n        syn_stage.STATS.clear()\n        syn_stage.STATS.update(st)\n        return zm.copy()\n\n    bp.predict, oa_stage.predict, fc_stage.predict_pkg, syn_stage.predict = bp_fake, oa_fake, fc_fake, syn_fake\n    try:\n        with parent_release(ki):\n            res = ensemble.predict(root, studies, series, pkg, threads)\n    finally:\n        bp.predict, oa_stage.predict, fc_stage.predict_pkg, syn_stage.predict = orig["bp"], orig["oa"], orig["fc"], orig["syn"]\n    release()\n    FAST.clear()\n    FAST.update(t0=t0, parallel_s=t_par, parent_rss_gb=rss_gb(), workers=[{"n": n, **pp["T"]} for (_, _, _, n), pp in zip(procs, parts)], total_s=time.time() - t0,\n                main_s=ensemble.STATS.get("main", {}).get("wall_s"), win_s=ensemble.STATS.get("win", {}).get("wall_s"))\n    return res\n\n\nFAST = {}\n\nif __name__ == "__main__" and len(sys.argv) > 1 and sys.argv[1] == "worker":\n    worker(*sys.argv[2:6])\n')
sys.path.insert(0, "/kaggle/working")
import fast
FAST_CHUNKS, FAST_PAD_LOC = 4, 32


def strict_check(ids, failed, parts, tag):
    """Raise if any study-level fallback of the package fired in the last predict: the stage failure counters (ensemble.STATS, incl.
    Basin's per-worker stats in FAST mode = fast._merge's {"workers": [...]}; Brainstorm review 10-20), the failed main studies, and
    any part with a NaN / missing study (every inherited fillna path: a stage returning NaN for a study it did not list as failed)."""
    S, bad = ensemble.STATS, []
    if len(failed):
        bad.append(f"main failed {len(failed)}: {list(failed)[:5]}")
    for k, f in (("main_sd", "failed"), ("win", "missing"), ("oa", "failed"), ("fc", "failed"), ("syn", "failed")):
        n = (S.get(k) or {}).get(f, 0)
        if n:
            bad.append(f"{k} {f} {n}")
    b = S.get("basin") or {}
    sts = b["workers"] if "workers" in b else [b]          # FAST: one bp.STATS per worker
    series_level = []
    for i, st in enumerate(sts):
        assert "nonfinite" in st and any(g.startswith("group") for g in st), f"STRICT: basin STATS {i} without counters: {list(st)}"
        if st["nonfinite"] or st["det_nonfinite"]:
            bad.append(f"basin worker {i} non-finite {st['nonfinite']} / detector {st['det_nonfinite']}")
        for g, v in st.items():
            if g.startswith("group"):
                nf = [int(x) for x in v["nfail"]]
                if nf[0]:
                    bad.append(f"basin worker {i} {g}: study scans failed {nf[0]}")
                series_level.append(nf[1:])
    for k, v in parts.items():
        v = v.to_frame() if isinstance(v, pd.Series) else v
        n = int(v.reindex(ids).isna().any(axis=1).sum())
        if n:
            bad.append(f"part {k}: {n} studies NaN / missing")
    if bad:
        raise RuntimeError(f"STRICT ({tag}): a study-level fallback fired -> stop loudly: " + "; ".join(bad))
    print(f"STRICT ({tag}): no fallback fired ({len(parts)} parts checked; Basin series-level [skipped at scan, failed, frames replaced] "
          f"per group {series_level})", flush=True)


def fast_predict_all(root, ids, series, n_chunks=FAST_CHUNKS):
    ids = list(ids)
    miss = sorted(set(ids) - set(series.StudyInstanceUID))
    if miss:
        raise RuntimeError(f"STRICT: {len(miss)} studies have no rows in the series csv: {miss[:5]}")
    if len(ids) <= 3:   # Kaggle's commit run (3 sample studies): the package's own sequential predict, as every scored kernel did
        ensemble.STATS.clear()
        r = ensemble.predict(root, ids, series, PKG)
        strict_check(ids, r[2], r[1], "sequential <= 3 studies")
        return r + ([{"mode": "sequential", "n": len(ids)}],)
    ngpu = torch.cuda.device_count()
    if ngpu < 2:
        raise RuntimeError(f"STRICT: the FAST runner needs 2 GPUs, found {ngpu} (no silent switch to the sequential runner)")
    chunks = [ids]
    while len(chunks) < n_chunks:
        chunks = [c for ch in chunks for c in fast.split(ch, series)]
    assert sorted(s for c in chunks for s in c) == sorted(ids), "chunks must cover the ids exactly once"
    print(f"FAST STRICT: {len(ids)} studies in {len(chunks)} chunk(s) {[len(c) for c in chunks]}, pad_loc {FAST_PAD_LOC}", flush=True)
    R, LOG = [], []
    for i, ch in enumerate(chunks):
        t = time.time()
        ensemble.STATS.clear()
        r = fast.predict(root, ch, series, PKG, workers=2, work=f"/kaggle/working/fast_jobs{i}",
                         orders=(("basin", "oa", "fc", "syn"), ("oa", "fc", "syn", "basin")), merge_crops=True, pad_loc=FAST_PAD_LOC)
        strict_check(ch, r[2], r[1], f"chunk {i}")
        R.append(r)
        LOG.append({"chunk": i, "n": len(ch), "mode": "fast", "wall_s": time.time() - t, "fast": json.loads(json.dumps(fast.FAST, default=str))})
        print(f"FAST chunk {i}: {len(ch)} studies in {time.time() - t:.0f}s ({time.time() - T0:.0f}s since start), parent rss "
              f"{fast.FAST.get('parent_rss_gb', 0):.1f} GB", flush=True)
        gc.collect()
        fast.release()
    pred = pd.concat([r[0] for r in R])
    parts = {k: pd.concat([r[1][k] for r in R if k in r[1]]) for k in dict.fromkeys(k for r in R for k in r[1])}
    assert pred.index.is_unique and set(pred.index) == set(ids), "FAST: predictions must cover the ids exactly once"
    return pred, parts, [s for r in R for s in r[2]], pd.concat([pd.Series(r[3]) for r in R]), LOG


if len(test) <= 3:   # commit run only: the FINAL form on 24 train studies of the check list, BEFORE anything else heavy in this process
    chk = pd.read_csv(PKG + "/check_studies.csv")
    CHK_IDS = chk.StudyInstanceUID[chk.group == "edge"].tolist()[:8]   # v2: the 8 edge studies (v1's run covered 24 incl. these for parity)
    t = time.time()
    cp, cps, cf, cs, CLOG = fast_predict_all(COMP + "/train_series", CHK_IDS, pd.read_csv(COMP + "/train_series.csv"), n_chunks=2)
    cp.to_csv("check_pred_base.csv")
    decouple(cp).to_csv("check_pred.csv")
    pd.concat(cps, names=["part"]).to_csv("check_parts.csv")
    json.dump({"n": len(CHK_IDS), "wall_s": time.time() - t, "fast_log": CLOG, "stats_last_chunk": ensemble.STATS, "total_s": time.time() - T0},
              open("check_timing.json", "w"), indent=1, default=str)
    print(f"FINAL CHECK: {len(CHK_IDS)} studies in {time.time() - t:.0f}s, strict OK", flush=True)
    del cp, cps
    gc.collect()
    fast.release()
pred, parts, failed, secs, FAST_LOG = fast_predict_all(COMP + "/test_series", test.StudyInstanceUID.tolist(), pd.read_csv(COMP + "/test_series.csv"))
print(f"test: {len(pred)} studies in {time.time() - t:.0f}s, failed main {len(failed)}, {ensemble.STATS}", flush=True)
sub = pd.read_csv(COMP + "/sample_submission.csv")
assert set(sub.StudyInstanceUID) == set(test.StudyInstanceUID) and len(sub) == len(test)
sub[ki.FINDINGS] = pred.loc[sub.StudyInstanceUID, ki.FINDINGS].to_numpy()
sub[ki.FINDINGS] = decouple(sub[ki.FINDINGS])
print("de-coupled:", list(DEC), flush=True)
assert np.isfinite(sub[ki.FINDINGS].to_numpy()).all() and sub[ki.FINDINGS].to_numpy().min() >= 0 and sub[ki.FINDINGS].to_numpy().max() <= 1
KEEP = []                                              # probe: every other finding is a constant 0.5 (AUC exactly 0.5)
if KEEP:
    sub[[c for c in ki.FINDINGS if c not in KEEP]] = 0.5
    assert (sub[[c for c in ki.FINDINGS if c not in KEEP]] == 0.5).all().all() and sub[KEEP].nunique().min() > 1
sub.to_csv("submission.csv", index=False)
print(f"submission.csv written, {time.time() - T0:.0f}s since start", flush=True)

if False:  # FINAL kernel: Experts' sequential check block replaced by the FINAL-form check above
    chk = pd.read_csv(PKG + "/check_studies.csv")
    series = pd.read_csv(COMP + "/train_series.csv")
    timing, preds, all_parts = {"ncpu": NCPU, "gpus": torch.cuda.device_count()}, [], []
    for name, ids in [("main", chk.StudyInstanceUID[chk.group != "edge"].tolist()), ("edge", chk.StudyInstanceUID[chk.group == "edge"].tolist())]:
        t = time.time()
        p, ps, f, s = ensemble.predict(COMP + "/train_series", ids, series, PKG)
        timing[name] = {"n": len(ids), "wall_s": time.time() - t, "cpu_s_per_study": float(s.mean()), "failed": f,
                        **{k: ensemble.STATS.get(k) for k in ensemble.STATS}}      # every stage incl. main_sd
        preds.append(p)
        all_parts.append(pd.concat(ps, names=["part"]))
        print(name, timing[name], flush=True)
    pd.concat(preds).to_csv("check_pred_base.csv")
    decouple(pd.concat(preds)).to_csv("check_pred.csv")
    pd.concat(all_parts).to_csv("check_parts.csv")
    timing["files_per_study_main"] = float(np.mean([sum(1 for e in os.scandir(f"{COMP}/train_series/{st}") for _ in os.scandir(e.path))
                                                    for st in chk.StudyInstanceUID[chk.group != "edge"]]))
    timing["total_s"] = time.time() - T0
    json.dump(timing, open("check_timing.json", "w"), indent=1)
    print(timing, flush=True)